# AP Model 3D Foto MAX: usa toda la GPU y la RAM (A100 / L4 / T4)

Foto → modelo 3D con textura HD, **esqueleto automático compatible con Mixamo**,
**pesos sin huecos**, **6 animaciones**, piezas por hueso y exportación a
**Blender (.blend) / FBX / GLB animado / OBJ / STL / video MP4**.

**Qué hace distinto esta versión**
* Se ajusta sola al equipo. Con una **A100 de 80 GB + 167 GB de RAM** usa el perfil **MÁXIMO**: forma octree 512, hasta 150 000 caras (antes ~25 000), textura de 4096 px y video de 1080p.
* **Usa la RAM a fondo**: carga en la RAM los pesos de los modelos (cerca de 30 GB) y deja **todos los modelos cargados a la vez** en la GPU (1 foto, varias vistas y textura). Trabaja en `/dev/shm`, un disco en RAM, guarda en caché los resultados y deja que Blender corra **en paralelo** con la siguiente generación.
* **Pesos sin manchas**: los calcula Blender (método de calor) sobre una copia cerrada de la malla y los pasa al modelo real. Cada hueso mueve su zona completa, no solo una banda.
* **Turbo**: forma con 5 pasos + FlashVDM, textura turbo y Blender en dos procesos a la vez (archivos y videos). Barra con **porcentaje y tiempo restante** y **monitor en vivo** de GPU/VRAM/RAM/CPU/temperatura que se pone **en rojo** si algo se satura o hay cola.
* **Ves el modelo antes**: el estudio se abre en cuanto está el GLB animado, mientras los videos siguen renderizándose. El visor usa un **GLB ligero** (Draco + JPEG) que carga mucho más rápido en el celular. La GPU se **precalienta** al cargar y los tiempos aprendidos se guardan entre sesiones.
* **Esqueleto detallado (66 huesos)**: dedos de las manos (5 × 3 falanges, nombres de Mixamo), dedos de los pies, ojos y mandíbula, colocados midiendo la malla. Animaciones nuevas **Manos** y **Hablar**; los dedos se relajan o cierran en las demás.
* **Por etapas, de más a menos**: primero la malla completa de la IA en alta resolución (`malla_alta.glb`, pestaña *Alta*), luego se reduce a las caras pedidas para la textura y el esqueleto.
* **Física de telas**: túnicas, capas, mangas y faldas se liberan de piernas y brazos y cuelgan de la cadera o la espalda, para que no se rasguen al animar. Los vértices duplicados se sueldan y los pesos se calculan con hasta 3 intentos.
* **Huesos de tela con movimiento secundario**: en túnicas, faldas y capas se crean 4 cadenas de huesos (frente, atrás, izquierda, derecha) que se balancean con retraso e inercia en cada animación. Queda grabado en GLB, FBX y .blend.
* **Piezas con nombre**: Cabeza, Torso, Brazo izq., Mano der., Dedos pie izq.…
* **Aprende de ti**: bajo cada modelo, **¿Salió bien?** con 👍/👎 para forma, textura y esqueleto, más un comentario. Si corriges el tipo (persona/animal/cabeza/objeto), rehace el esqueleto al momento y **recuerda la corrección**: los modelos parecidos salen bien solos. La memoria se guarda en Drive si está montado (celda 6).
* **🔁 Reintentar con lo aprendido**: según tus 👎 cambia la semilla, pasa al motor que mejor te sale, quita «Textura rápida» o usa el esqueleto simple, y vuelve a generar solo.
* **Revisión de UV** en el resumen: textura aprovechada, triángulos estirados e invertidos.
* **Malla en cuadrados limpia** (como *Smart Mesh* de Tripo): retopología QuadriFlow (o rejilla de voxeles si falla), UV nuevas y **color + relieve horneados** desde la malla original para no perder detalle. Con esqueleto y animaciones (`modelo_quads.glb`).
* **Detecta qué es**: persona, animal de 4 patas, **cabeza/busto** (mandíbula, ojos y cuello: Hablar, Mirar, Asentir, Negar, Sorpresa) u **objeto** (Flotar, Girar, Rebotar, Bailar, Saludar).
* **Texto a 3D** (como Tripo): escribe una descripción y se crea la imagen con HunyuanDiT y luego el modelo.
* **Textura rápida** (opción en Calidad, activa con Turbo): hornea a 1024 en lugar de 2048.
* **Dos motores de forma**: Hunyuan3D-2 (Turbo, segundos) o **TripoSG** del equipo de Tripo AI (más detalle). Blender **4.5 LTS**.
* **Malla reparada**: quita trozos sueltos, cierra agujeros y orienta las caras.
* **Cero vértices sin peso**: si alguno queda suelto, copia el peso de su vecino; el informe dice cuántos vértices mueve cada hueso.
* **LOD para juegos**: además del modelo completo, versiones al 50 % y 15 % con el mismo esqueleto y animaciones.
* **Visor web** (`visor.html` en el ZIP): botones para cada animación, cambio de calidad y modo AR en el celular.
* **Multivista**: si subes espalda o perfiles (o una hoja de vistas), usa Hunyuan3D-2mv y pinta también la espalda.

## Cómo usarlo
1. *Entorno de ejecución › Cambiar tipo de entorno de ejecución* → **GPU (A100 recomendado)** y **RAM alta**.
2. Ejecuta las celdas **1, 2 y 3** en orden (la primera vez tardan unos minutos).
3. **Opcional pero recomendado:** ejecuta la **3b**. Genera un modelo de ejemplo sin página web y muestra los tiempos. Si esa celda termina con "LISTO", el motor funciona.
4. Abre la **5. Página web** (o la 4 dentro del notebook). Verás tres pestañas que se llenan en orden: **1 · Malla → 2 · Textura → 3 · Estudio animado**.
5. Para no perder tus modelos, usa la **6. Guardar en Drive**.

**Para actualizar a un notebook nuevo sin reiniciar:** ejecuta la celda **1** (cambia el código y conserva los modelos cargados) y después la **5**.

## 1. Proyecto (código incluido: no necesita Drive ni subir nada)

In [ ]:
# @title 1. Proyecto (código incluido: no necesita Drive ni subir nada)
import base64, json, os, subprocess, sys, zlib
APP = "/content/ap3d_app"
os.makedirs(f"{APP}/ap3d", exist_ok=True)
_FILES = json.loads(r"""{"__init__.py": "eNoDAAAAAAE=", "config.py": "eNq9Vttu4zYQfddXEMpD5F1Bli9xXBcq4FyaBkiaIE2AAkEgMBJtayOJXJLyJi32u/reL+sMScXybpLdfakAWxQ5PDNnZjgc3/ePmGaZpqSk5OTyhjzh4Gp+DgNWFksG/0QwuShKUlFFaKk5+dgwktF7FnneIa9JU1MyH8QxyRmZxuTkAPYOJvs4gBnEUow0inagzud/np5fzDzNc65ICb+K5wzfIArwOa1Bg1xSXGZ1a5yxcs3+Cs0WwRRXHmwQkhnh2omiyiCj2YqhAYIuixpMf0/6OVv31arqWZYVLUsKUNTT7FE3khrOZF3kjBNFSwAzCiXoKZusAKrOBTTyfN/3ikpwqQkY4UZq1eii9NpPoezn1cXFNUlALmL1upC8jpZMB/78cnSU4pofEr+f8VqzWvepGOV+z7u4uU6PTq9e2wbLX+1KweYipwp2H5wd/350/Oput7yFcA98cybbN4B4Xs4WJF2KJujNPAKPlk92gI8jqbnMVpvJhZ2IsianUaFSuqZFSQG0xWgfAcZ1RMG4FMJTZCwVkkOa6IKpIO5t7ZEMwlQTEdW0YiG8Nde0TCtWcflE+mT47t0oNjvYY8aEJsfmBZHb6BZUKa8D5h9e3oAj4ih2hCWt0rxQD0FV1OlCMpYu75Opsx7CfkilYHBgILW0pPf0A8e0Mzm3STBVkBWFdFKCQuL8jN81DwngZjzC3PnKnagJXGJTKEIDUjgzSxb4Larfi4xQl6fzuZn/JSEdk7e9nQP0M5DNFme8vyUH6VLRB5YXUgV5CG4slE75Q3ItG/ZiLEAIFX6v03ObkoLqVfSBF3WA+Q9p2BpjdXzbio32TeByU8naPLMpsoZogkqbxHanmbBHM4KDoZvnDAp6NqG2PJyJRlmjYZRmvKlBBeGSDM3yJy4fQohalboAPqcPHCAXHWMERGc/nll37Nh6aYtln/wGH173XOQF8LClMvFtrYQU5ZkGFcneYAgKNRMq2YtDkq2a+kEl4xifcCtENq9oxlQy2DPLBCodiP40CcGDeMyTYTyehlDTypTX6CXj4xdg+GJRcponv9JSoV+xSKYSkeMp4CpaiRK+huPQuIRJlYysw+Ea2bhgNJ11HDBGB7zOfH523eE9RkO/4j38Bu/h/897+ALvYex4n43JcPwm6/Pjo9MO7dF03NIeb2hP3mQ93ZC2PF8jbVn9AOv9YYf0YPICacVmX4DtkOsxGey9Sfpg/sfpYYf1cG/Ssh5tWE/fIj1+hfQgxuD8AGmTCd/DeeCO+A7BNqhqshVt251lQ6VpYrgooGeAu6Lb5MB9AbcOVEXoOWruMNa8XDOJLQdjEqVA1l4Y0Jdgl8NkVcBVbZoQVy8N8oG9sR0MCCBuCYpI4JZM94Ua36N9IegEwyoBaPBuu6teZBDErY8lzLRPqTFY+Xe2rmEiT+JWCroudFf6iRXLld4SGsdt6XMTcAw7l8Gt7zxo9lT0MehOhcQVDhE1IqeaBRi0TTnH61hCGc4D/ArJoBeSrel2djvCWMcT/LPxgxotk+3ivUFoZxBm674Rz1eNymRxzwLhbhu3Hix87FP/Fre7YPTu3WcS4NiZjd8nBz3y7z8mTXClu4Dz27fxwkcZNBolMIAg4lpoXLFDXIN5e3LMvB26+S8hV1RB94Ji5tzs3s3Cz9hsQ1aBdNsK4zqMEUM8vgTT6X1QtvUpbIBL/D/sVZwi", "engine.py": "eNrVfWlz28aW6Hf9ih6kUgESCqYk23E0YWrkSHb8nmxrbCe57+qqUCDRJGGBAIKFEu3y/K35Pr9sztINNDaKsj233qhsCUuvp89+Tjcsy3qZFEl2LObwx84dsf+LWPlR5IujU2H/Vsab0o+PTvcPxQNxuFrT+0LeFmXm4+XTSMaBzNy9vd/zRARSZHJWZnmSH+8J8b14lwRJLiL4v0oCiX//KmXgx2LmZwsf38lYQF/PL34XPl6s5QdhpzKbh5F4efK3Fy9fiwfQkhAn5+9eOyPh56GIE7H0N0LmUM7HFopMijL2aQpig83k4aIM4YV0aRjn0FEqYVQilyKFIWLvser7zclLYc/82VLi+FN/EcbQKlyeh3F564hIbmCOCY0CZ+Bns2W45qGnfuZHMK9joRrNYCBqrqLwswCGIhclVM95JGeRKDJ/6r+H13KVJlBdrH1s6kEg1w/y5UrYQZjPEnwEI3PUBBhENPqkkAspZkmMXc38OAA4/qteCGH/CuV+oKpQJgPIyJiGrodKNU0QiYWMAZCzMInr1ajhlsm8jApaK+h+UeKkqCBCnMHGQxUwG1iYfImwi/Rq1GB7X+aFzBFSqSzCDIuswnylCspchHFe+DH8k4m7Z1nWXogAKnDAgFMZjNWdl4B4UBQWaDbX7xez6ipKpvoaRxKF1W2Y6Kv3eRLr6yTXV5nUV/myLMKouttURfJyCtCfybx6Uiwz6QdhvKgehKuqnQ9hClgs9/R9XK7SDY48TvfmWbISFy/OhXr3YuUv5Ij/vE7zPS7gihoC83Cxt/fb2ZszMYFRu6lfLN0gzGJ/JW19709z/Gt7HnbseY6zB0vpnb/+9f9CrWqw7nkyu7advT9enP35Fl7YFvQWF9ZIWFN/do1/Izmn+yxcLAvL2dvb+0a8BUTOQsC2YxiWj0iTi0UIdASrKhIx96McJ5eIFEgcUKIoV1OgBxkB3mTrMEiyPZqfC3TtvXh58vzMu3jxt7NzHMLjh954PMb/e/j27YtTnOfh+OET6DqQc5H7c+lRt3a4WjjHhFaAI3/4EQwJiD5OMmBb4Qef8JIHiCuGb+3CX/lxMkLChN9zLFokjos4hu2Ec6gOKJIzBs6oD7UaLv1WHeJP5odAB9BvKc+yLMlsC2ha8QScPQDDGMGaxgcQxKo3I7GEaUHjbh5+kLrvG/GzODoUSD10ta2v86pliTxqBfwQSBOYG/BV6NVehXG4SrC59NZR3VIX30Pjv4j+BfiMDhcZMB5Zd7Dyb22cngOd6BWsm4UZq3nPknRjO+YLt1gCosR+GNm2rjmq2nD0MpyfvPr176/fVh1iTeSzvHIxIPGb508RZeHPieX09x2vZVbYqgiVyCQwlBhfK0SbRdKPy9RLyiIti9y+ljKdPBkD7oSxN8+k9BbTycG4xsCnwGR94nAGpwQwiXUo3yPTDElaVaBTtFL4gvh8hYRFtqkHHeQw5hxIXwZ2gAgrApwk8jcXf1Uk/z4JY5vZg/v693fe6Ys3AIPvLcdBIOlSYQ6swg6cGvA4E+yCeB0wkvzaK3OkrmZbjksFQfR///3RuKp9swT+goMEJBA2SB47yHHlEViIx1TpZxNkxoLQ/KDvAAaXpPbYabxRI8pWINSlHYxEuADSlp5EdMwn77JSNisY05S3YQ5LFoD8s1xgv1arV/yBsplcJWtpFmuU+nzQyNuZTAtxRn9AnNa9pz4IDUawv0DqFl6ULHK7RqLXM8Qc4a/DnHUk5CTA8mbAjgSoCBVqAS9LgNe4QH4oMKDdBMmS2gnCeZlDt0Ayr569Rim83BwFQLoOCX9qVmJbCwBTlohVKXNUnfIEtAJ44a+R99VckWUPjHSBEs54dONnMTzKmampGxdkTiEzfWtbvHBAkiuQmAC/ieV+bwze/V5zxTsbmPmg8iTZZvKMxP+fXIJrI22gEGQeoCaMfEBduvnST3seydYz0GnxiYExauLuQhbncCkzG/tx3Bzu5VpGti7w58mbVy9ePXe6ZExSHJYF1kVmuYsIlbfgigpBANc16cMN9uEBr5omeVhsGPcV17wDx/qHACpnnKPY2zaKwhxF8UWjYEz3VkBoWeh7pJQoyHYH56piekhYuhZIzJ+5hTu6VmUBfff2ZhGMQ5zFMDnJJWhEHojHwvPsXEbzEerS88mrJAZCgulO0iyMCwMBsJCbAivAcsjWFAMIZAGEaUgxKggtQFEThvQYmM0UX2A3zTeEhcNvvNV6y8tia8WB16kPExx+412HMTJmRf/Vy/fJFMXRR8ufFWj4WMcC5KElY2+WRD7efWpW8MDEAul3A5WegVZY9/UNKMho9YCAlEGIstDGCzAEknUYKTYFTAk0DoA2GJ6FkpXN9iWwShwPKXLQ/xMXx6OsUrg/ekQPpmwSwYPHY7czRmiEENPQp/uFqRqyi3aDISoaiGwOzS3TABiWjeUBK/zATlKQkM1eHUMWD2N0g6yrPtjoAgB8aqOaH2w6MKdXChRemiQRlJjN3XdkDVzA/dktmOzAmG3Q4bybJLsGJjE5AoWRMePSUs+sq3rIAI6Vfw1Ll+UdgJEU9pJrJatr4pNxDtybAGFQGYibtyGKn1kShIsEjUzAsxJ1eWWrGt6DjV95DkAsJKs0QlyAN1VzfpGFU5gOWCKlRCudcCpGcxusXJKBmQQuMAt9MkFziSKzEnpaOeHV8oJwBuyifoMsEebigzC2LUWJIEWQoJytpZAm7yhY0yDKJWuoGFIjFNhGjUNVNV1CdcKRoXIAU+rhfvQ12KvGeWjzPpQ21B6hP45vsEdAgTS5A9rFYdgpAapkgOLetkx6sUCVzqlkS4e9bBa7+hLC2mP2uP8FP0wZFbnBWiPz8W4k2vC5EnmZBABPYPoSVL24eGA4+Cxa487j1RqWokGu56D2FpVjr3KHofOMfGxEcYF872dR5eB7c/KyQWMk+ZclKR9zfya9ZTnVCkA+8wGZcZWB/rK6CtgcKIcur4Z5MGqDGaqCzTZsx6WJd80BWPKM3nkhWVgDxaq25ZpKuUhIyDgGytbj/QEGDFq2PXfJJ0O8n5qac3drep5fDUoDVM1kj6QBTcO2XhG/TEswDCIpiaFVnszfnh3DkrasJaUjjfdqsgBcyQKbBtbsJgZgjxtPbkKQlyTNsDj6h6bQxbSczyXoTovJ2MHRzrswYYMRBUI/vKbQFQsw+/FD8fPP4rBlFhrrhQb/dBjsaoZxb4EY1wPt1alTQwBVInTcufjLUOtotv0UffCYZipvm+MoksJH+ZqXK1vegohM7QxMWFphp6sv2nPrgilGeY+VQxyI5Vh85MaUfXnsHsw/iedP8bXVndrc+mjMQOyL4tgdzz/lhhRRcKFWDaHM4ikNU2lwiBF6zkYCKkwTp6Wut40oTbUVyzgN3z2LkpuXfgGMIV5cQNNRaCic2BfA6K7yLvaGEguMF7gN7BYeq1HOkwg48ITGC1aWh45C4GJ511kAqEMTai7ZN+IZmArLP04B5IFEHWQezigUwVGQ/zgY35I7J/PTUEUBBtmPnp4rYx9kgzfHptfByl7NPD9aJBNrNWt5GxS5v9uk7HHrNtjby2BPXbLZgaF0GYuowIK6UxDmaRKH0FEPS9mx/Xu3rU0/mGWNr6RPM6LegHkAehWtsrojFcp8QAtuPvgLFctiQ1afgdmmU8YUdb8iVZoaqCveYZPCfqR8Lz9Uk3HQQwp6LDqZKRYC4wU70cA3CoqASiVyP6KoSRUbEr8C3DDCw35VUfiraQgFKFqmVV5ovGpKhZjY6w2jGfNwtAmlRvEreZ+DhuTVkQr0xNQskPwy6P0HhSZ3Z2XgA/cqVmXkAhUkN14xPzoEokVIbqkUx/ctPgWFYwnW3nW7dIPXsBGtRv4UWlhkCQDuDTnyMpPCzUXuV9saJSaVLgbj9pLYW6QlKHLoU0oKA4XuFhbQt2Hv9/bc8Ad0ZtFsq+6afKwNy3+4cSIvwlnAKmEGbxXWMvoRK/Md13VbzKiX1bS8DmwZGWKjX5cclM9DP9ZSVQ/2g7DYXx/uj/cJAMAVul7fz2A6DIK7uFkPijTwEuQD6DooqBuaaBv1movWv2QaZYx1RazrFmz06/opKGCBrawzR4sxnt4GA7vZWh4D0yDaRzZzCPwo8uOm8JJgAG4DWC8aMQKNdOM9KNRxb+2IL33r37VTNZhXa4LwMJWQD+wzCAUaVqSy9rOQojh54eefQSvKC7fT9MHGuj/B9NLMav11iaYGyDbCaS2L3SAFYKYDC+N0V20XStFN3IdY0Ir9YvSH4TXR4i5CGFTd2m7e/1Es6ceTLnF9ocKI4GkgyQht0zJHhYRCTAeU6TGAOuybrvCBb3uRoaEZcNimY4NcYPWO5UEjLsmZbV8aMKHOTIlz1RLBiCzA7R3Q9vrrWVcdN0SODg1KQci34Nm8D9Fo9ML++B31/h0OprqkFnlA37H2992n+2Lf4ItOpKAfoNtNsw7y9fP7hu3Wb+zfbRsZWGTx0ikIHW8dEqU5fFUI9E7S2dvevA64wHj7vSKZ9K+/FnnuovhsjxSZnLiiz2EmrHga2FXsOQbbPPzAXgZUuOGhGVHodHBpJfM5mnvW1fEdA9VGMBlF3iwtPVXVvh/PBx1RZTa+OGnCS9gz4GAZRgI3GJQPEiPJzbEa7Awh0mdVdD2VKQWtG7Jg1BZ05gOn13+ZmstAHLR6YqqXAz4Et0hsC0BmOf3BpIbyS29u/GxVpk1zpRK4LRc5epgiDIY59/BrIDGzcovwGWj6n6NvtocFYvPeY/qnSPj7S/km6r9UATbAam04GE6FCLQddk0qjwXbE9ZX0x1W3e6B/rRPtIdb1cmM7J21Cygriwlh20gEPhjVMTsAXdDUsqLFCurOWbulGbqU8GqMRUMAFZk6Z7gBA9fq9etqeNaQu9NBq2tj2oRtxE0VvXXDpr/HPg7PDzCRNg7BsPcjHmZ2TMHRMGLf1bXMYuB9OKNffz89ERvghOFCGpkuuUD3ZBYWK7UI5emrV5xtDW2kGYw60xm9PiYs+xgBwnTtKJnt5mJSbt+GxdswDUx+QoWV9qe9Mew/ItdUmHv+2ofpAWtusxZ2G97pr+mGjvpSck8z/6Yp/1bQHmckxvJGpRKOgFYODkcCfjlwPR4J9c9xOllvVbMu/rLDVauIK6MoTHNp2wc/Ph6Jh9DM0RFcHI7HDkUSookN1yNx8CP+egS/Dh89ancE7BtTxxYRtvMTVD/4CRs6wBZ/rBt6DA8xt/HgcW8z6KdiqD+nFHGMfARyHc7kxMJ1sBx35cdg5oGUBx2oFbKh8AlXD2Pgl8juSETb0D8/98simfk5BnWxOaDbAnS9Cb9ECV4cPHb6Xew2ZaVOwtUIE6u9uoe8kGk+gdVIZphC6GUSjI8SedHk4PDJAHvFJmbAPq/ziRbdfGtdjVR+PAxqsoBWKS/Uo3FioHkl82XbM1Ajar6JZ8sMdIgPcogBzS30+wLXA1mApLwLq7hXjFDoxleY7p+IZBUWILsUU63YDCppXpGoJOsqQWqVkgoig8m4yX1QVUpwC4hKD6a9H7wIoe+KZ+e//8092M9nS2A9kQCcPXwIyMf8dAR8cUYFqxZPUopbHmJmATrBK8v6XwVv+ZDAxkdaHT8N37nirWLKyPbQzIySHbgQzQj5UJHZfIMCIgtBobk8fjge14bcvIwwlqdKYbIoACJEmhKzJRi1s0JmyJuS6XsgthGXnybBZiRQrgMkMPSHmfWYCnwz6onbWSBCQJ24WYaFFNPKIwz9JHPQ3ooyCFE0LZbQ/NEpwJe2eOTFBo3rJTwHqBfAB5sKKMhBv4Dp8RqamQ79BnWvIsZptTJehzADToo4uTg69d4dvrAcMZlU2obVL905k/xNGSMaq1xyxAkEofBnM6BKYXMedJFcM/78xvkAoLHMpNOjXDTTKfUCP4vK215Tf8Ch08bMghBZp7kP2NJKepl9dW3AaQRLuA+qK+DqPgin/EGzr89U7giDPZM1TjVv3GY+rTN/5S2m1pX4ZSIej7fEEMkEKKttCnebTO3o412GV7USgIu84QTgCEiJbXx5MBGxqtfxZHCLAYO37UuC8SEWtPxJUH87gulpaZIY9dTsd5IAGyUrYF9VeFBX3F8fAOqcanSHq7wAqS0DMzUKPQijhm4FYzGZAbsYgFwJ5Md3yne2BRtyHSxsGy+cneV7d8GMYK6NjBKYfAnsnQQ2iqfJGBl/nyQHqbGkFKbJAYqQrRR0EwbFksvhjhQYPm6LgaZA/i3gzeGjxw1x7ri03PmlwfbNkZqsv5e9M/+uObelZKUBskbo2qPUtHyhcy2SpJhYD2YgI9ASfIcv3z63WoKWnwr7j5O378QDQffi5IVzrKLJwDmJj85DoEKynQKZw0P2t94tE3tEhplAN+zlUaBqRnnbwgQzrGCaep9OvuEUwB6a3ujdKnGO23TGDB9nl4KNvEKsBZPIZzCLIrccp6vnq2VwU0WceXWlV0iDSkG/l/6pqWkWgoFmhKfh/s3Lp8+btNLOMlZjrKWHztOzetREGpKXqcCxar8jfhrto7qBpfYP3IcAAYPFu3LtR/ZQL+Txacz47n4qtKVuukqO0u2bWv2QNa6QnexmYa9wbzIieIXpFfZbzp24uKuarGfe8XZ2x7VVzhydDmfccVJ2xQyUBdzgBrTpkJgec5GaRza2szEFyxXvLt1O2eohmynN3ArifJ7a0Frtvchk6mdqp2UzEU736L7yVzJ4x5uXs80zeGTn5Xwe3k4sN40XaMXJSBZywo4nyghskTvtgPTXIAqWLm2qaWW8U1a+ejfMWdIwos0ZxpBVXuJ04ZG/dhKnrp9l/sa+PED5on9dgQWK5OTFspi0SMzYIxfGfhS1Oq33j1HG5N5n2so9MnVneZpUoh4HrQxhAEbLUu0TptgtbtVUWHYIFv9WidoS0kS8tn5IgkNf/yIecVTqR3cMjCD3MTe/T7AqZHTf8V87gTKun6N2a8NyURdH6EfBtctRNIaLMkH7DtcxuTy4MoVqf3wBDbUPcqQMpnayVyeGMSTZjKGT4TURCD91o31XoGnzEzOa0e9dagYtXbRkcHWKkJRobkVmLoJF3Wm6fAmP3tCTvYEQiEqmN0GBaKK3GQ/X4I50hWxrJxUc6vHYKqPe9LNwsZEwBzPBXz1cHxcPdWdzwZwvToaeYf4cbz7nvd8oOtQGhrbjQ7tcWw7WriKjWGROxwP0vGnyXvxZo7eQK7hgSOOuFt4Ut7GHPG2cekdlPMylw72PYE7pPGDKYh1yhDIBjof8THhGQ0nu4o/rlQtGRdDILn6AT1tJx2NOOv6v/+xLOubEY/RefVyoFhRFfKdsz++usLp1r61OX2MnhD6fwNwMkZpSllm4N11wiKDBG3DD+Bam0NQSsHzr2IC8yJyOuKu8xpQ9j0cYtDaotw442Oup+zrNcYvz3KPtnKA4tIoufcCGKF36ar87bY9H04+c1BQZrOQhVrx0XXckjq7cFShyjvgZxMHYnFgFJK6aFHUPffOr0xvt9m57Uwffvh+fFz+TMyAooF/ARXSv0bEuaHVlSNN88EnpBxnGYXGL2AqDPzW7RgD0zrQqsQHhd5tzsTiJP8gssX2QYQfjhicNdwrc5q3VvAUd4vZgJDbwd3MAbdzmDMERXYF0hatN9WyjnvVAjICQJaltY5sbaheMzQNqF/6aykHP0RFE3koo1ecvfA8KzsETI6z0V3/EApXMwWhF/hdwfFASGKNtOxf74gaYEDAGrAp3S75zOvnaUBVEAUpimw1x/N05xoGEMAH88CGzrfwvQztOo7Dw8qWUhZbneA12XobynM8rGeEhJSO0vUfqmJKG2XwarsEeprM4lsl7sh04aYvP7Ykl7s7JUXekbReYsD1P4qBlL8+5560MwdiHmS2mfpMUhii3degB9dE5hkTNGjkKA8ngIfQONK0hOmsQAY6rogICe1VuRjtCfdePN2YEJ5cLwBAYCQARR+BnBWW8qnjyyBAzmMoQjgTtjJKgdaIKiqclRE7HE7CuWAm33ON3bvXIufthK49DgYqb295UZ0OsOUOdH2hTd9CP06D/vpb7qiGfwPkatf8koJKZByprA6wIx5xzxMgjgo8Q0aAcEBbqzLBCfwIhj93xw6s2P8LixKyPiVWiO31ZAr/MjzH3H8/iAX0ZlPW8s5cLaxMBuURftjVq+0b06Mg8CWEEf4IqEDsjYjHwAHlS9ZS3sYW0jQ0PdLBjpx4sWm7UVKKjK1Q6wdLtMdS1krJo7mvWpzsA02G262ADH6A96mBEI26bR2VxiW+vqux+FP5IAQazVTycniKzIrOikyuhCBxa/CpqCTkxatMFDddQ+9xQ6WenJW6HzHfddWWfSmX0SYyaqL0EI8E3QTmDmx2DDs/Q8pKZaqIGAo6MQh/ma9ux8XmnVO9wOoUHdesNlgNttvFyhYhUvXLR9ngr2wkdKyDKIPDIpGyWpb5d5JFgjOcMaZeB3GkC34XxwlOqT1ACkqIG7unq9pYquJoeaBMeGBwh5jZ6EtZsS5VZBJLAWyZArCjA6QJNpcPxeLylHy/JMJLLiOLNkiUZkVHbpkDbY4X7bOjoNIaM02u3tNxDxnK2jfUVwVHewhThFWk6K4Kl8UC5lCYtWrpnzLpawwGXmxpZX2pQbzhzeKp6ugBh0CKHVppbZmjwUrvz8NbjFNy8hd+DFaJIL3enfC98DreHvsj1xSf3cVSfjdvDhhCreAoJSxQCBgWIX+r3x30YYDASRcY1lwJZP6k5VptrEqBbG00Vs8N4eF75OjlRo+3zVNtP8dA2MHibJhq7hdG0UR5Utm4+L4rRiy9q9ma2kvLV0tgv1ZF5V0MO206D32i/erX5LwFlUDAp0xI+kOhk84/pBAz0vPuYxFXlhQ+F1HhJuirOvSOpenxzdDvc6drW2n4NIJRT9j1W1ml7w6mBeyNIx4Lvc8asyqjAEO3H62NQGlGruFYKK/XkhoVcAcWzimpgSq2J6P0trEZRe0g8B7UeDRjIOwu2ZPDCg6tqI7LaVrA9//fKEf9oa9F6LGpbQmsIdOKAgbdmk1s6b/RcXLXz9mJ5W9hpnbbMfbXTj2v8wL3YBC5O5bszz9lgWTltnOqrV2yrNqeaLYmCCIRyEGxxcrnjPcioJw6r9/qJ+HkCNhHD5NHnJXMnxVJmrQE3YdoafHcu/Vk43G4rxbvxFEHVn6RBxXpSvO9K/TBcjHKVFhs+SsNue3GYrGCQarUJfk0G+c8JkHxBQqHi9I2cwi3RE4VBgzp1N+sQa/FTZ7Q10XDvs6IwTjPsMzyw3sTFngANCxRe0a9i9ii/eyPTUB8EpQ0fxfBRtdwWurEsFWzBY4QRwqT7hJiOEaOLJxLLJAPJGSQuu5Ym1WHVvOm5493ZKRDUK2TvHRSC7uz2BB3xL5O23oIhEa229CjI3ZBXp1GTTMkKb9LkFe5j40fXVywN+fjEgeN/SSySnMQ8wGtnywE834g/eINknQuaC3ue0UHTP+CZ3X4Eq+C6juDcclC38RwdESXCD0CHlW1bvrLiffJz0kJUdJprrwhekzxmHoS3gNqfbX1opEHncmPHJ53LGbCuPeJtjTy5HkPkzsE3Rlj7Gnjr7tchvUQdWYJ0dy03TfUqSQvT0YBsUJ2ajVLpwGD3FY7QmdFNuKGQN7Gj6/Pbuk1sqc/Lu3YBXYibO8OF1iCxppsCzbPL42Px048GBJeNk/eCcpXmNk5xRCfpejB9dSRMT0cK+kt3KW+DEKz2Ajs4OLyqldSsjBX42P1MaV/6+EggGHVJNKOuyUfL111/r7fVmd2OUY26eo3qhGWLuiFTTF0DEutxYE7aweFRK31Oi5AJnecG/U6sZYnR0jDAA9Z02p37BDOxgEVxll5z8aFoMrF+9UHD8jNrxA9Q/OmuORFADwmAXGCwsmdOiqWp01oyGYVyjT0+MRV9ci8sMvQvRP5qGvgCRcixUKCIPfSBhn7UA3QamXJLUFGg54WGG+ZRR3pYbF9OjHRO5LAsQJRb47h9miSfbehsP1HGVChpg10vB2qkz15ULGSvrdRW6giKnYtLi64bpzQSZqiS6oaL8o1Zlj0EXJSvuSRdN1IbKF6PxfCKC7X2clY4oArW91y8ujcrKUTR81J3amZ81xjGuLWxx0zxJwSxAVkPQYj9exkWlPhN4R3LHCYwLT5IlOTiseEuViRe0aCjD8c3CxG5mmW6yheLUrMSEXqzYaZ4s1DGOf91qU+9Ial+rmqat2ximOE0FVKqOZCjrd9e88Pk7P0qUgOcqFCAONhrt1DpHgNqVufc+WeU0qo/I4EZgCRmjeVD1o7HiIczRQaThrtgor0GzBLpN/NE+N9O0+1Xmyv+WLsckEnC/4o58p+KP/Kf0QB8kFXSb5NNVlc1r1R/+5upWeg0SSK7unWqfBv1gm+cmpWy3aDunMHWgdPqlqeJM9J8k57xtVPzS3qqbhzNOOn3QPs1I9WjVPdmUrvmeaixmLpKI7G9Qm8+8ZCs4sZ59/VLwMpLCx4BAxnS+d5UJ6zLmM9QRI3uujeXs9FwzXjanw5wWnGtrYefEivLi2xO7Mz69v99u/o28L797duX3771LDx54rr/+F1SGFtn7tZ6BqUe0UcXGr3jh0Tw0FjeYuQui1VkObS5zaiM4hGHrSQl8mJbCV2OyQXHLacPHmjanugFn3PqEfhHQ7PAIoPToChgn7/u+J665pqzTpsZzNTz3CL93vsYf6IkVscQKGcaHaVxEPU34g0634k3RWBrjsW3vDEYLRk8ZRi//oOJaKii+LnUHzfCz6VkYZIZx6Ihbpxdqi3vaJvZZ5fVabvssWNaVvldiAxn1eGzVONhDStPb4TH6DG8Grs/jfHDQ3UHD7BPowJLdF2xqnHnKLidxkGiRTi7tsEwmiKDnWGwLaeMqCRtrRWY4SdrP/5ALH6KSQoiL2FtVmQoBpjqQ1yfPuvE4MsWTeudgHcwsL2XsJEOHSWFCwaAuXI5hg27aHGrPIQw8UcjsNGbuy8PcKaUNEszOugxTyqVw8fVm0ItH+PktzT/ng0MOJwbPyygx0dOE4IpWGCyC8J53OZc0Ebjoz5nawCa3d4l3yjS3CqPazUScJ1PBpasuY++1XLvtvreWIrimvO4Vbg3v7qCT94J8ap+iWoNqFVn1F9Wh2BfoRP8wJQW+jtIwOxn15hQ2co8NHTGI2CMZzHmxfjH+ttkQLNo4GDMJlEHBFgtx6TuoUfAtAe3bw6uVUqf6t2awiBw47U6b7bB+ohlGvxyjZpV3mdZkw41REHaP0ruwED58wlBAU5AKopnjAzusmu2gcVeYNTLq2Mxj04tpB+rffiaRYbTunbvWANH63R/qLn/+k8+w8xqRT+gmV2Hy3Lv2DzGY1us6/MP8RiKf/asXMXpWwwrH+ZP1YJZ5+EqDQn8mFmZ8Ur4i/K9zJLc6otfAh/kI3Grc/dlzBvPDFesjR4nTLtWOqE6JufFSV+LF3SuBGVtZxRiRv0i517oGIoDYMtRlMR8Lgh+7I++qTftiYZ+I/7j0VhQch9HxTPce4LfXcrTkjL91PP6aN+Nj3urAQndntSxeeLR3CbtiHmfkVQX/wUPaoCfzzh4a2uejxGGH2xgx4g9D3DriVu7HN/SzUWALgkH6CsDfiMjwdkaYNeZUDzWpkp5J+hoaeQtQqmp2JFibNFS09q4i2hqOV/vyBrCIVr07TP9Rhw6BoprhEREzBVWp/g1FHSUE4Jq5zN+4hI0SSC1xLmTpElvGxNTtmj1Q/xoJRLOxwo7j0efoNuPnOwB19S51YvR3TSRnhSRr4K5O2LtAPENrj6r9ZafzYCDDK79oIHUV71lHzXsJJsRzeIy5uIDH0x9cXDc4DsYJpX3l8W1En48ZF3jywBGca3UehbXDSQ5GLHST2K70vDvKbOsizAutPg2wyUvTu57zEFDturgYDss+CUiVf9QRJBhqJwOrAcQx7nP2Qc9cB5Kw2FJzWXulNO63W0YzdqnCgCGHzDU+ZnIPdTSdjzXCNPEdI3nh8eEBLT78h4nvg1toNo5H2G3DUaNjUa7mSJ9mnlDfzcMa7S5HzVt7R/UI6Ws7XUpmR1XhEbN7hvmt9F0/bBuHJ7tGa4J/THgeSkzDour7wSjgkUGDfAgzhpofgqYjjN0jZZO6esUmNCZ5NUXmo/BLAclj3DUrz4AM0K1jwwkjkLjt1jZX1g3WEyHjk+P07bj6A4sNQZ5UR+2X33MUB2QT6kBvHLHlCvPIxJxqOP3eCjaZmbmyxOiU9jc4jliKJzqsR9ENcEJVLrI1eAn0EzHaauQVpJVIe1HrSMUObqTOUSBX3aizy6BfQ7XFg4qAMaF1zy46ogFGn8dMZCdVj41fQ4SlQlbeSuaSMh1AZ8JB9ELYjwZiVtnb0ApQY4HxtfY/elH4HV0jy6RRm1yjjRGsh4eybozkvWWkeh3CCwqny9MAjlnHx1QAbkcJR3wl4BBRP65/BgxSDlH+XPGwp9mYVYjltGWjPWn6rSXIPI1caA+7H4dd2z1zSC7oZVsoRCAf1GCpdY9kqHt7WQPaxXfb7uaslk/bc7v+hIr1Oxxt5lggBLteVYn2Zhf13Lzcgpatt11m6hSuEclHXH6hBGg0ZEZFcYwAi9VuKUVhdbxlCpugguj2QBRSrOC1l5sJR8rnV1JSDMqzby+ukUTxdwzNO988RHPHFXEXRF2O+EGz6xULOnVawqT+fTJLfShygXqVcDeZffkSrAvWrrKN8JfYBX65OzfX1wQXgcVXsdNO3m+3nmh/qcWa3DBGGYj5idtk5I/ZU5wYMmGMkCLzFrGvX76f0bi7btzPF5JfsDD3vnzZhtx/vq0XrPb/x/RtSAFjVMbmpOf8ndK59LlbwpvOxKTyv4wEdY/4n/E6Jqb33Zr3SuzSkOZRtiylwtvGnWV5KmpaNUhD0MfMuMgWh+aRsOzqrKD7LOR6Pk0JwDwxnLuv2dd+42aO9jCjl7DfglYdMwvTdqfFe0y0bAdO1GdtL4vCsWMVtLb7d16tAK7dH7b7v22r+/blv1BUzO/t6i1RP5wo87UCm/5aMHmW6ebzuCx5IuSwGOvBR6eRGSJV3+VfpCrF/1pDNxAeDtgf8BoL689RKvwFi/umAt9phs6vrxyAFAFbgXttOt7NHJfdazmadTEYcFbpbRRJ50yzsDZMwjwBtJpgBIMyNkzwbMjotx2uNXKCYSKBeVtkwowUY59nTzge6E/YWtkVH2eyiNrZkInnNmFdiEfDDjPq1qqxaoetQm16hKKdKsS02iXZvGICl2lySzG3H43iaG3Rf99CapfPuEcwVRm8zCaXFxafEUnv6YlPqCM/xreNQNBoCoeEqbMN5CxY/gST6DlpDDPz2dhqPPOFBJNcE+3kcqCW3O71Krgg7wTW8b23ZsshHVG1uz0pTz99ATDV6vUBy2E/TMgyA0S/0D8qKRD7dwPYdr61rOHe2up8w9pT/t0gJJ1jodzNT4tWCXiBGE24ephOvmQjgg5Jwyna7mZXGuRNqdPuU/ma/2AFPYJ6+0tqwCs8zBfeqzk2FDT2WEH11oJurLoyqyaTa7vYJPUZ5tNGoR3Bx/exljXzgAzWvc1unacIda1C59d381nB1vchdv1c731DlzvLqZnDOvSqpgA4wIpAroX/W4gNmgyz3W3SIOovwZNG8RxSdyXBxvPu58bq76qugMLwC9+9ZyoVgnLpWIQlc62bnCKHYj9szYP8nr0REAGPWvaQicfxlX7SxMtWzFbYXBVmZ3hIk5gCSQmCubmV8YVpc23ZAHN17RNHDsFfSGKMIXTbvAXQ/Hr+H93HU0DPjpRjE7XyoeO+28lS2BJevZvCKZwtpLFMgnq3bTmyhlnVq2QuwOLR+YON1YzRQEq0Ql+fw9TOrYPShCCj+o3Ly6807Nn5yfvzk75A+vI9iMwK6LJASHeh26uKXklMMWeD4fEoxsx/Y6M2+66Y6M4SKPHt+9evzk7paVz8duQOFjbttxV+hC1O3WqoKX9HWxDgYre2hlprlvvjDqlPyhi6dLdHM/IryHA26bwrpG+lkmwpGcKkEhD9WYBCSJEbRcIlIk3nLne3OH0RymjtQRLNE3wk/WVQ+GBjx95QJcShQ/LWDs5K2euOg7J+E5ogSdboqsLv/gQkme0zEs/g0s75+9dyyLMVIzeFb/HYOEejfcfj0Xe3CvVddAGu3hnw95qg9bPoGnTbxjxWhvmls88VqvSrOG1PhaPKU+4UZcscL04PtdgM9zCUyKfONVTNsrh6dgd10+1k6Ha/dCjZ5pFKeN9JH48NNrQme34DQanOltHd8sWPfX7xOlV9zCWZ10kMUd7K1Sh4+5w6T/C5D5Zu3rXmh6K7xFQpverz5+w3f8x1OCwfwI5YtM/gcxsmwMiJi/756NMj60c/BMtZRj+/x5DGQY7bCcPmKfmsJoWJY6obS2SQ7syT/GuNkz7A8sGlVu1+Us11ZcwtUUHV2jKddOG1aRhtXsS7OopKy/B3uDSmezmEjGIFBv4a1a5tGZJlslFGCRAK5mn+HGnaK2Rxvj1iC/XSAf0zGB3LbOpXU4mqNHs10wHHjCZIhE7PYYldBVUhuc9zEtSiQAKek/kb2FB3ymKsCfTe7v/i/DXdB4t8EYXNBZKHAt87eml+i9fnJ+ByvHq7C266mrJq/jcsbisszkozVG7LY/VYcM67YEL9XsXmoEAFz8PVDdDiEKtUJxJlxtqCQZ/i5ls7ICm5p6YzYEiOysjao9LDDZEDuxMH3qHLR0150cF+KNVg62s/JTT82r55OJ3iupm9ItB0HB0LNC5tB5pNgFB6hEO6IwL4Iyenz8V+vXO7d3IqdIWXfy0kYWNUK5fwh9Ih/e7NwZE6c6nt9RYa3DPnv5th3Yq/uv+iGhwQbd04uCyBIDv0oJe+B9x4V9ipkWFDjvPA2QCNvHkMTTxB2+7lrlKgcyFff761Nm5MaWDqyO1rRMWSwYpWo6RUaRUpR7K+gNfNOhK5jMZD1BVLwYfGu3UvChiJtLfigr9V/o0EQLO443OXMCR0KDp6wpVue3tLULAr6PHY1qoTnPV24FGeMQaon9wRE+TogLmp9rCUCxbn2r7udEkI4LU47rQe3Afjzj4OMFDUCwOaWprptp6Szta2yc2zFb41bNLtd3p6fnZq9OzNzC/fTLl9vfn/qxIss0+7SYoU3pICRdjurzoyvy+yLnWPZFU0w2JsP197iDE1SXwwA1AiMOynVbhJcprBT+4qywB/FIU33CzlTGAL/jG6WuuikEy7PUTZQVg5eqeG65tAXyp7nqbrtYMp3ig09r5kUptZ/Dt1yyaRju09w9KquSTusX2qRR1AXJy1P30NEYHWKPCJjx1SaE1u7dnwixLYRiNWiXP1GPhDWj/MqmEdHXSRpVoo4dTq2d0biAVqiU+adGqo9arOpXHsEbw2yB07qmhrPAXe9Uxd+4FaVSA5jiBAFBrYr58cXFGz2WWmc/fvjt9/fu7u3IaUfNVO9Kn5ZyOBjyox4Y+LIyDNhRtHDDqsmnm8mg6uq51QmeXinnmrySlFVAdVZd39bAHxnqGMdoeM2FFrixQfdEasbNGi8L+R/CDY42otV5H+KrKglAchL/6zbsNJ5Ws6Hfi1rVs3LFFG7bcI9Ap6XwNF7/Ck9oHDm7ZekhfTOxh6T0OYvrMQNzyP/KHYdRRrwScTJ1l2s17UdyPprJFHzHBPaxo6FI9oNcd2c0c4zpbaePfkbDUbmngK88dZLhE9ui01647QnUasH0tNyO9A655MvAqdwY//Bvix8oCfcoVtKFhMbwJQdEBnU47fPqRgTjtbXlAK7QZr+GEKG5xWybYORYLGkKGhv0P1dgywbNEkDeNB7IeuLXL/SPcF9I98YVf7+3teWevnr94daYXZG8PxTzwK0/GC/yAGBScpBkgugLfIkqmfiRUtT01LN1KZ5993fxZ3V4VHFCD0Y39N2M9qwA=", "blender_rig.py": "...", "app.py": "eNrVfduO3EiW2Ht9RZjCbDKrs1iZddElpdSgWlJLMlrTZUnds3B1gRNJRmZSxSRTvGRVKVHA+sELGPDOwIaBBRY2ZgcwbAwGDexiH2zsPBiQ/qR/wPMJPudEBBlkMi/V3WPDPZgSkwyeiDhx7nHi0LKsl1EmkhH/wC7FkNnPE+4HcZv5ImQizXL40YcfUZbEdC+KMzGM4wsWMy+exmzGx0HEd2b5MAw8Dvci/H+W8FREnHEGV3ORwqssj1iWRwACrp+Ece6PQp4Ix7KsnWA6i5OMjcN4qK/fpXGkr+NUXyU88uNp8Uvoq1R4iciKZmk+nCWxJ9LiTjZJBEwsGhc3gqnYKTqmSTOewtXOziiJp8xh6hnMZhSM1U0RwWyFfjQWmSvv7Oyc/OLlqzdswM6sJ3wKGEmsDrOexEki6OoND3Of68tMXn3Og1BevRazOI3x6hWP4hQvXvAhPjzfOX397M2ztwh7scPgvzuwBtNhIvrMnvE0Tjss9rJEiA7zOKC9wzJxleUJbzus12XUBBZrKgAuwJjGsIoxe5snw5jAWXRpAbTjDju8f9Rhd7vwX4cddI/utzuyzVOR8TDkPrW732HHvYMO6x0vN3zNZ4FsdQDgDo7vdtjRcqsTwAC2OeqqLu8vt3nFr4IptTru1jo86j64C81udp795cmr0y+fvXGfvnwNc7P2kfSAVvdf5NF1zqPDp3sH+zxNgTT2xRWfzkLhAtCxSK2dnR1fjJg7CkKR2n67T73GiS8SWkWJJ5dH0N6PnXE4xEVRd9/n3E9r94B7aneSYOyMhlflHWcYisg3WsiVCj4UPdAgiv90M+BDngVe7MTDd8bbxe1pFhq3gTgTkQZxdOg7KT6pAZ0F4gMvRj8TQB/6B0+8AJZZ/4QJuEE0ih1kRqBEfH2ERHQGr8x4NnHexUFk+x02arNRnLARCyKFw2DEdCNxFaRZai+/0y5AfgYwRyWIFLhL+DbKAwf/LL1r7VrtdtucGPQ3QuGErwNAkBP0awQs66eXQTaxbcsJ47HV0d2CwMgTbKxJIc2nU55c2zhlRQ8cJksYAFa3Lf4uTzOB3Lm4kZ2HwPuEkGIo9ggoF3CIUg7+vjzps93dBcI4axF/usj+rfN+50by6+4u+/6v/yOMxs+9wEeRaeFkyl7Lt6w2E2EqmAUXnxU9jizVATWXnbQ6rPXzVtvs5OP/0IKBLbhsCr9b7Rs2u8KHlgFQpO9zEYosZgZgoAZsbRu3JjlSD3TWhQfyR4cRRTEQkBWQxlsz9VJrD+C1sW/jYSrGOcj41M1iEDlqHjcsNch4ZH0RJ1PeZ4vWNA+zYA7kxVsVpJkPFNJaPaCvLG7dVIZlN3U9QvDUK43urUIbrmVlSq00qHWrMOwGZbcRdtnYjWpcdmQCfw10K5J+ZQHolkZcBSHf/+Yf2dtAgFpK+4yGz1ZPrMRpZTLMJk3Qrk8JbxazgbmYhLRmVkUv9Zl9HtI8Gl8ehsUcm19e2K154AsgsBb7jKVZYjdAkS1abeASaJYyW4BIkQ/ZLAQFS9iGOVUmuvR6MWWrwmtEmNuQ7O6uWiMpbypMTXrFAyEN4kTJmkKeOHw2AzTY1knZCCgPJoNCWcpAEigVKOdKHnoVkeWFPA1GYJlhK6utB+KVXWbBjAyDhTXJp4AbmDwoXdALCQh9jj16oO2SfCZIr8s+yYg7AuMi42ldwZCS8fgQtAy2l1dsnw1BeJKJA1oMRAs+U1c3BQAQVWMazdkFaYMLFOc2qqwk4mQWDRP+QRpIHoiokADKYdByejTtCyXil1A6QktGeBkoXGm+gugkBNBrnqJ4uNFqd1jlZ/sGlnN5ovjfZyj0mT1HS3mByy+XSM4FJBzJczUzLb/lSuTzymLl83KF8nl/1RS+/gbkST4/A9GCmuSm3S8YkoPZOxfehIMioSZePBQJiaSZl0Fb9rMqR1X+G1lZEnz6XTTOQxxqmgVgF8MVQSp+ViAF0RzgB0Wr8nfRrJxTOVMyoFxSTSvJXytSYF6iQBpKCNZNwA2dakCqKlXpiYRARNegXMNAzMWqeVuTOIkEwZdaOwZdBzZ8yOxf1S2+X5GCy/inf+Rs9wmOK053201zDGN/wwytL796CnfGIomXGHynujCLC1Qi82KCQFTEHx02RxaR4qDs8dwJMjFNbW0kVQY25eAUuAonqwf3WjboM2yPiIlQe4C1zDz0pdAdTCW6FPU1IYGoARwxUAshX7PUX3z6LkX/EQBiy1QpvrNWBYBc4fmn38t71PrTP0fw4r4Hg9wH+TUGJ27VMocBMEO5zGCkSLkCFCLFCgg0YCKwfWCuoDGIaXMRNU1MWjubpvWCWulZFZMyXkYWsRGHxjyw6yEHReUJ6VEDULDMeSOZkTWFqwmqE+e2ZjSnhWXmcWSMGdr7PCyGVQcl0T3LowznEIw+feeht7QSvzZKCLAAxzxBEmVZDPY0DB98fgSNaEjzlIGACtDJXTmZgJw4FO29NnvMeltPCOx7ub6COlVwqtNTNwnvIC2vrtn00++ApkWCrxooLmk3DSIXXwbUIIEgkjXTrR9aHzzwLghKySZTEmdpACTFOEZXtJG9UhzbdSZYHgnOQz9mxV09DZQRl3qsyvIAq5gU6Nn5akr5/u/+/n//z9+wxeWNAqScJevb6NtISSh6pQ3O0+uXzyk6YZ3kUq3zXCr6F2RQ4J3StMCoSBJwkhf9im1RmhHWk6rNUFgR+PZXdesBQyowwwKd+CgKQIlFaFlI324eiEuRuJNsGtr+4BdgMHXYRATjSTa4ixEFfK5uXwaJGHzBQU93mMiAG9X9aTqmK4U0y7KeyfAYO3zKbIwvCfCT20h6OeB7lPApkLtIwJo0RCWI5xlApVAaeOngpaDvjzEwRXXotPqggOhigwdtqQCdg/MCm6JcTrVcsJKP/GAOdvJ1KAYtOeP+Qv57M7t66AcpGMTX/VEorh5yUEXRHimOvicwKvhwmTIt9IGD0fWeCrXolghhzwfceRlYmn3QvPk0ejjms36vCx0NuXcxTmKwk/t3ege9w979JthDih3sYTQuT/u9I3iRVHj/zgP+wOf84Qg63RvxaRBe99NrcMane3nQetwAypw3vZUGH0T/6O7sqvX4T7/973/zaB8aPMZWj1Hx2rC8iHXrTT7EUCUnZxEUwywPU86aOhhq8GqI4r54IETr8XMB0ponSBYou1+eAPhnmjgf7Q8fo/cO78sB0F/NrRhjzJMQ9BQsdJIWgcb3eZwJavIeGA3MtAEIhszGGKaDf+w220VRA254QUhI0iVBvMcQy8j6i/lggQ/KZkjutWbWXxAP9Io2xAZLjSRzIPJoeDb9lhNJE4/GuS8Dqy6fBfsYaBss6iTcMkkYvKafL96rsRkkrJkp8QatBfy9aWnUXwZ+NgEK6/7s4TJxS2rqd2uU1kRly+TZAhEdxpdAPHkYpmDxiOghu0r2Uhg+6My9LIFXQMq0asDla+VLsMJy+DUirXImWlB7xH/9BIf/EBQnWKB7WTzrHyHJPuIMJMyomH8Gz0U2aIHHzKOLVo1ArSpl3h/eH45GD0s26B0izJNhEiQor2Y8ovAuukSzEMXT93/9t4/2eUmdOzt32N6P+I+NFVOgriZCSQHiqyAFF4yCEsBraHaF6MLMWAweayCdzCKS22ejPJK+LuhQHCeqeY/7aKsQOFsq2L3HhTsEl4VaaDvsCbpFyNu+mMPNOZojpRQGZoe31X7GHbAl2BTMtAy9qhAgJhTVB3hcTsBhqHXgcZROAwy3IibnwTxmUQDNkXEpsAvwnZ1/+dXnpCK1PnIveZDZadZhEZAGuBQJWC4iTU3lAh44RgMRLaAsoKMhfxczsM3IfsCx0jjYrxDEr9g0xrFiiANa8zkaj4VeuZwA81FXOkSaZmcW/gZXoYiV4j0f8Gudl7yuB2bD7O2u8+BBh5qNrHNwkEGreQPwTeBOa4qGCIbyTKG0h41bWRcMSac7wohT24g7QLs0FGIGcI8LqwuhgylmjiHhAYjCceI8S5I4QWcBeQysVgQObaFnIlEZxJ3FNpJxicl0El+6eh5ujNFlWNs0QAeXo+1EiwekJre+cOG5PwU1WKplKYWDCNbEy+RGwQXuN9RBK7OQmjlg6EVo5wgbhv55nmVx5Hhh4F20UbgD7oGmUlPULZbh9dkZTgZQi/H0Cxk+KKkIV0saNWq6d9jXan5o5ZEOIpIkcpmLD30kDk+EecgT5eMAfflolUZsKqZAs9wckmk3oQFkPSOSU9zgAxO8jZHi0uzT75D+P/7hEIlAW0Zq9+TjH/sMIwpDUKnRav9Bya4TuQkB7Hgah5++G8e4caVYMSrs6HLFlXFEOwoSCzNYm+XdBnNes4b9iVmbOEHfBfGKshJvP2bHYMYT7tECVB2D9TFzaSw2KO4IeBm1SIeFYgTXJMaBWSZC6H9c0jvwCPA8F+5w3CEY5QZeHS0jDkY97edBSyFgFsjgrrwc54GPTE5daWsWfsQZzwRatL6IlyDSXXAeAGpK22EIHhCS8WEo9M4hgpS+P5BgcQ0WML8QKji6BNinDUIxeJvk2BIWOhlYhexmdvLpd7gx2LZI1k1n2cCyqD83wR1DrizuOljNCwNgoVMtitqGlCRy7AF58QgcBkNOXpdCvnC9ZCQJPUJacmHa3QVmS7mDv4CS5J6zg/+g0YXblLu7hz0Qbj1JUyIaQ7NyM9guhBnRBQtSIhuiLiIEfafsShn+tsQNGaJW2wGKD2a2YdY3SMOawYq86CWBuuXTj5kXfPqnyDFkbyHVu063p+Q4uGWC1AcgivZII+W1IFKBLuLv/+q/meI7ua6OS851gNhwsL2bxXKvVU1KknAJQFx5Aqb6jP5BCcxBn6+d6sj6BUmsWY7RWxhtsnKsoBzEjdW8DmUf5pDBDwoyl5bHXmbatvSd6QUF9l2AxKESD5wsvhCROxFX9l1lAmfkEo9AhAOScXcW/dJT6aFHPrmtsARwU7qXpPSKH6SG+2yJKZTOBrVwjq0C45WsC9eG8pVBdTQ8zmCo5zjWrIgJxKGPqioELrCxSfusv3fQNXQu3nRm8cyGlh3qAuStZPQRkQ/IPOBzgzaVWYApBvzKVr9A7oYxz6CxsVuLz6ayZQkzjlxQx0At0iDya5BxqviGX7mrLBgdt8CfxjAv4+TC5J4lkkUQCZlAkgaSPPrBonydSlOdmaJey3Zy48RVWwv4QqpvhIdRF6JKEv/wp71KCRhSf/Mo12iFjS8rTTGQa65+0bZMqU+kGkWW2AiOdh4HiCHCXZs9GoCTWxrLA7zYDEZqrMEwjsGGKVUYoEvT3EBfbAam9RxBUz/am19TChHleZyOLWnoJjbdRjnPk0wmKVhvsUmx0T+RGtTaAlmlKqWxlb+reRIVbioYQN26jXDWeVTgBwvklqVOyIqnMMAiu54JW7Qd10UWdd0bKZ+XXiFg9FIB1pHuIVh5nl2dyCyRpFG8Vj4e4eZN2MDt0r0B+GinSElRZIY5b+nKVq41Sg+QQxw4PCKzRq2TGkXpwVnSHjWcOCmBFH5LxBYRDSliQCp3qoFJFYe0JgFRiGHZyj4oCYSSczR9eCG/tlTMkkyv9XSiQjbKatonW73YZEPg6A3HYU4Gg1Wja9DD+cwHuWKn4FB7mfAHauq1hrhJ3tAFKhu+u8vs9S4i+OmnsLDaFFGuPBof1V5o8dGiaJv2uGpuE3K3sx4P+qWpCPaDSpDD0J0CVpiJpNNJM6LchS4Mr3XZuqgbam+VZQpul4hkjqQfP2SqZ65DJNpOMwhM769tT2KNZAVgrE6VBg7kAr3VG3ibVzxrblqueRXkLVa9FsI6jaNAUDij2Ce5ZkbCgzRHjaUnpr/Nwh/2S9gdEzR2lHgTcJzT/+dr7+K8XHdp7RNTfEtq6NAuD3osiiQ6eIX3VIs5mawRrIxtr8vgU+l30ARXD0DZfk2JaLu6orqkmWFJN9oaB0l8eLdrFUl5dLvirbe1ZYngPqDX7mNk3PkQzCTSKTkTbttnH2bnDS77h5kShGfnGFMvkjlX8wJ02UTVKgpdp2ozwNKp/tqpmRidiriWqFAocKazI4AsZ1Nnm+//879lu7uvpNBBdCOFbIii7e7SNhzOuJKzWIUNeisA86IYPg4xEYgNd+qrF5iL2S0u97IceIfu/QTBXj6TqWq0y5WnOU+CeOfty9Ov3rhfv1xONTpdlWp0siHVyMgyerImy+grnWX09UuXRoFDmPfZRSWFQo9PZ04UQbYlDEk2R3lgr0y0Qhd+caPcRoRgdYw9WLDWdQqQotKid3yOfK+RUoS5aktXDmJ5gDIqcYWh6wGzmC3XQ+5DglWdgnJLEuFJYSfTk7abSblDW4B0YQ2LtNTaxs2dO3fYx//1hodgTrAhSPOffxs9KzQsvOFTMhY8LHKxbuAPjfzGYd+AF8M8jNmDz5yJfl1DoIr+02//w68pvyCJrwGUjN5iXhTIcEAYWTQzkeE+R8gECmOnRCnO2Y1HRfZ3xUFU08DUZwf8GRCH4GMub8FWc6R1vk3dfl4CW8RwAYNz3D0jrYWJhBkHxwSIElAi4Aewjhqd3hxerWVOExAYSay0CQNMDBM6d6HSz81txuJ8gxn1VdeOHpGBodXDU9EqmXwYR6l4hzqouqMdX5SjLtssqutJlr6MkuRg+5MJFiMR7+6mYooxYbQjTuYY6gNa/v6v/pmdhIJnOGQYH9lu8AOakxP15jk0X5lcYaX5MKAA0u6uSGc89JFkZiIhKQ1vUjT7FaaGUBpx+vGPTl38aFuoHDKG0T/+Qds+MujJP/6xw6RBWYTp8McUkAADyD9Av6OYzN0gjZc6qeQ1vAkoxAWzUzF3kI8dYudP342RqXiSBEPamwK845rRsN7ngTSo10TerY9/KHaplZtLoXsYOqJKKHucho97hjGQ3tv9k2WkoLCDoT7DxLZitFLiGDvqu7t0OGaMnKm73d01od2QyCmpzqqmooB8OUFG9zHYp1vdsEULRMLftMz9Erzz6xYIFNSYivo6Ba3jIYoUxWrhSIyCK5c8VuAAkq15sIVJWWg6mqsx9z5KOqUSYfqYViYm4LCSRNJTX0oB+TNyOQwOmE8rQ6lx9DQNzawMXGlSGpJAD1MjMBPcm0iEYSMZiyqHT/lGFAY40Tqjr4ZK+XbCo+zRVAC3RDofF8YCmkD7BLiKywAVwuoyS45B61ya6+0UdVM4Z2RpAu6zYoAKQ5G4rIX/nURAvzgYis0VpBPy6RDEzAiIr1/G4EcqAD9trzaEfwJb9rY2dPVtiXQYUykjgIqFN4mdJVsUENJex19DnnkTW1rDEkMqTrmWxxRj0GurmYN2Q+bAhjwlUaV3nFfs0QCCYQxhjA8xpH52XoTIgRnIGRJRDuyGKKau2w174wHYnSEYB7JBsS/+RUy5h4Ca3s3+omxwY+7EwEogc1AaJGgPwFbkCaSJNEvUwowcjJqtDmMnRvga4RFSB03xYcLzgP5W/TnEg46ig8+IdHdebQE4MtIDwWkpUnmGdNQTPBX81b6h00zLD5OzFkBtnVdmv0WYsdrvf/n3K/s1t3wUu8jVreQsxmNNhdNrl6RQapdC/DkPgXMpC1lLqDHIdJ9ySm2yA9AMmgcodkkTQstRghlw7VKCo/9A9FTQkr/5UJs8Zup89fVbPMyoTrjhBgMmKMjgXntpv1BDCFLppFdRh3GGIMpLygmm4638f3TlYaqunKozi8YW+bR4j4KXdKvdaEtsde6v4vjrJODpuBZgRjzqtbfhcYctLb3fPuv3js9B4M5CUKm2hYESixV2uCIEAnXWv9s9V4uPtrxcflvMsz4KkDckQJ+Cm6nQCCilZfR/yiXcWYErvYIUJOHRtb0ZicV6bdwbKFTAduuqDs9grAhxcAYYcgIwXq7Odxpo79a5qfqM9iTPgrDczqOfjhfPrquw9A8YCi25SxrKddtLfeCuoWmzLG6WvTr16Kf26mY8Tf//DLH9eaNk20XGfpydUg+C1QNfFei3CIKhnJhgQA4gK/L9gALhtkKAcN0sCLSeAtl0XMgm3ILJE08UiqlKwJJ7ZmmFe/RmXJx4k+LmfAqjlQ2deZDg7Fyyyq6NnTTUCPSa4+U+d+C5i1YSYsGmrVvjYZC6fI41C4YhxiVpNTAfpmto9CYrq+q1LRSKfJWpYgtn1r5BJf365BVo8fnUyVPhg02FeTbdvtMb3cCPZdcVQEFbeRyzaIwB0uef41mpqQPePGan3/yMzns+P/0aXDTu834zLHt81jtne2x81gUKq/W+oIf1XhqP0o2sIqWUDqKApzERMC80AIVDvwojqNEAulqKFuFesbhConRfvHzzlkKomc4CGc9yfTlP+FRfG5de2QITU+m6CG/KHX9Na8vkhQSCxxQzFB4zdHW6TneZLlGqlmU3yBa1ImD2gO+l04Dt7YHbkFzvwWAHCDr4wHG+DvzuSJqkRdfXtKid9XmCOBU0zPNEEJhZDALM8RN+Cb3JXeOBl847UTwRHLM1ojgHwZCu30xPJyIMVRqbx2cI3Y3zbJZn6iYGffQlOOLwbHCIW8M+XJU+KhoOMg3iSkr9KyqNoFO6ZLqRDaap1QZy0z87lnFwFf0i6GPWkVjn0K5D54l24ZqIERNt+NkB3gbxws8Oz/HvUQkC6cHNrjK5C6/e6nUPjois9xf4cnFH0rS1QdMZIC2MdgFXyVc2Sps5D1MyKbzMRjLA+QGwwRzTGuGfgl8B8fBYgYJLV91HfYKLPshm5emmC7cjcYvgdey+HC7xy9mFW+QIXZUiCXcniudn/b0H3comKgLsFPOFqymuhPYeKjqmdCASjqGFIhXcftClvc3X8buYcg/9mPFwjGVzUnkIKAww1RjYZ8Kv8bxqudtcYa/mwWBgRvPvzhYOlS55UTl7UDlh03r8Ss4MtyrxdBD8AmlPzpU6DbDOn34XD1N5n2cosyJYZAtvYs0MCzQs7qliShwab5GL88VfqprGly9fmQTy4J6ikAcHkkQedCVpPDhWhHD/UCXbgdcm63DYFhKkkorwz53j4fFw1MXLn6GFaFvfgJaxtKzEFg+G94ajkW6xUzeZVXPdujd64N8dGvCeyP481Z/o8u5hdyW0tyh/tRzG9kej4aH/AC8//sMTzf6/7LAXMBs6mvZAClu0MIACvAn4tNVgBfqsWdXflDl0yBqA4Q7LyeIkHJW0gGGHgvpLMT6n+jcTsiQnZ3sgMR4PcGEqrfB0D0z1+Oj+kU/bR/Qa2QPQoenV4YOqW0fDrWbsGe1RSU5qhz5p+llKe1nKjwdp9kuQgxh4uf+AhFln8QJ09wu4iScVrkhStkm2SR2+VpeoeM9VNd4D03/Q7Z/X7HJaiDIk8WgWh9d4FJLNYGBZOmgtYKw3LQxUhYNWBMYwnlZK4gtgtQUe3VkzkpElW+7RuSZof+AcV9Hbc+7etPYfG843UYQxnnTGI83aM+5jZlP/cHbFHhQnooqTTwfVk08rj6EuWmqtDw9b1eG07hzwA//wbutGHdSDGepzVz3oNI1DcIhWA/ZWgWs9XiBt3LBHw8cLokNSUIv8Bk/QPdrHWT4uNp1C1LcoZmQYV0uWwjLFwhDY6DHrGnFkkxDJLA35DRlscajPsv9ZkbuMWBrkCsTqpksN1bnDm6ZjkNCHSj1JCZMFjlpKFsvaPYhTNGYJLLYjXCxjGpSVS7SO9sQjurrqDVrdFrs6AGr9JVD3NfxeIBt2nR7+PKj81Iyg5nJ8rG/t+Tyd4LHx60HriB0hiVOPaLrhRnvDcUuJjcff/93fszd0nhg3FqpFFGiJ1WlLOncJqJPi8tvSY6Gt68rKqh6kqIce/vY38piLLICgUbKz+qCteZzwoHtwcDBcZorqgspR1Ze+fmSwgcQ0Fd4HwHQDT/nRAT9WHJB9WDuhuu4kYuVEcP2sL0zcE3tDkV3icUg68nuZ8Fkf/9BZXzpeu8BFu1nqpAHD2uZAnQyOn7Ju6ByZvFN3BUEE7NcBr3YC8Tja7FL+/KVaNtOIMQc3H1OW3OfxFdAz6zIgZrZ4ASSLxdREMhcndKrrNbotpVhfcQj1Hh55rvchj3QiilgXkKR56WbRUgVjSLMgM8Jgtl8jhfbaakAHGizJsPZN7bzxzs6TN29oF83aGcb+dYc58rAuLTbHihNsUTkWe693v+ezfyE9RA5u9U7DG+CS7Cl8HHcBC0b7h6xChjc7dyZ+wiY9tigPph4gBTOFqa5qMmOL6ilw3eAAUcmwWRr4ojbcYc8/OHrIlvhOcZd+UjAT4pBpbsIz6wgWk8L/HHBxuM4wjPGQgepF/qx2JmVHBYcKukKI7LOyKHfwNIwJBYmMJ3u0VuhFPej6YtxRhnHnzj3vyB+N2k0LNRqNzNs72lgqjhHfqy3wsnyqPb7aAynvx5d9ZLGDI3iuxnF8DCNPgnEKLpGDBMwWirBvmL4f8iH8XVQ7OagS2c3OKI4zpETNJMiqVQTJ+J8GZC7hIdji90frpnRkriYubcmma1Deuy9xjut5yDuKiNrVYeEGdsqGdFAUGCmIFCN1K82QX9U2ah6EvpsH2v1c4ZKdqt3Bmcp2F1OK0SUOXabO5zzFI1oBRlDdSS4GVhD5wRg3zyORZwm48XQ3DcEmXh87QcoYnJWgn8fxOBRfwF1bVplFj8kq1BLq6ZRH6R66Jnii2EnNHDOUSm6JVJcMakvJIXi36bnr8+TCaFQCQ+ZqgCY5DKE1NdDgdKs6OKINl7gFWxEzlrCMpwUg1aQ6SToup6FI6ajnVz7SIPTz2liIOaqQpLQsxpMFWSia+zK2R2Z5tg7pjQ1WY52I2U0FaAgf6athAbyD0SENst52/Ug3gS4wvgX8laj1JsK7AKmlsNswer3k61uuXP7aa82EsLJRbdyS34eCUtWNkr1oX56eUcDDHQ8t8ugP7yvDV1b/lVUCgmyCguFzJJfUJnoZWCenjLKT8Sw5ZjTgsV1k7wH9hdGl6QBMiUbB8C4dWHZ78Hjhx16O+YIOErWDKTjpl0EK3pfv2y2cBRa0bGO0yhdTo6iSHtLr+NIWoZi6gT+wwCawalvd0OQVQAHNgrGBO6w+6G+jpV2H00omQqWj9xi7VOWB6Ahmrbfbp0djiQ0sSPXhfR6IxI8r4HTfT6hmjp164AwMDjsYyHBlIODwoIv5jWr+KWVn9ZcQruG85cPUbnhea2NbqlKNtaKtynHBA8mkMDCtEcjQJjIcWF/J48Wfvhua52ViZlMyL6Vgtrc4uFbpDTfxJ3EIcnNgPXvn9BlnUwE6dA7W/UWECyJnkAYhBg14Mo2TW/aBO1sDi/IjccNCyDyhDvviy6//kk4VrzxUvL5uwYpdimsW5mJMRTkOnzrsC1nDhKvsT4HlIsbhp9+nzi2nQdW3Bgd4Kv/KlT+O2ish6JPOsIov6Uy2WsMvKjksyNrXM+D5WRDeZjiELUqnGFivn39+YhXFtQ4Ou6sHpWnxxPNAVQJWbMvIt2W2SsvtFFm5ADZ+x7GGAO7aN7Lmqj6Qrzc01TZcA5qeyZFU8bNm2r2DNdMuVlCMmpbklCaLosL5afujY8yrOwSe+wk7lHUOljt7AQuIBCfXuNbfrYC7ulY7dPE0iWck+zci4Uye4e/QyW465E1oQVNUPpCHvekJttmGDdSb+E4Jp+gAb9GPOnzrfIuzyzxE23vVmLUkBkxEqsggcoh1K54rMzrt18G44K5axtkKipL4f41hAJuqCWDaIeZ9yIGfyEqAVNdGDra5zsfaQ6ta70k30FpPG6YlYD1K8YDqYxyFOmvCmSpsrA78maeLOrcQ8Oro0T4dN2L2NMfyssPYw9MRGGy9ZrJBm+VMnkS6DXB7FMaY8j4OEo55I0P6NcTUi7bzaF/OagMi1OF0uUBPlAlpy43rQqwt5fvfUguZKvWpUJVVMYEcMTALUGQrdMBNH0yFPOQ/RI1qvNy9q8rM0w6r+IDdHRy2b0fwJ6E8nSF4eAtil8JJovNNCCZYYncd0L/HHdZz7slSPgNZT0WhdyqyJE5vN7ZXWBoNZ6XL895C1xlFFtYsuirrS9OnhNYVZXw39FJDBMz6wAFUdJ37BSpKTKDzH4HZyv3bYeMJDBFfugUS5BEgQyqdrShDZKlDQsC8aHToWg7bSOUmEd3ciZr/KxoU2ll0xOmWPUj20sPFsnfFGTo+BHdCmnD0nJ283LB8FM7OlsS2+shMIbnRiyyGfwq0Qp++uLV00JJBSZe+/O6MPP5ERQGOewcgHHrHXdbtdmVpa0CgoNNlmHa9dC58W1FBHVG5symApEoYIKfvGv08NGge6yn6XH0k54d1KI904fuyQLfD5Bds+uxYfYWno0cTUnXj9esEA6vZNmcW5ukg5eJncvBf/A4OnvKWS4ZJYmoZz3D5zs8Oz9u3XTC14voEmz27am8eqCo0UpM6qoZuDaI6E/eD9ExBUC+oXjs4h4gRUxUAYvpqLX+wlqEP7wipwvAEAIVasOgkuOWRzN6Rj6NP31Gl+w0IuqULUpQUWifE/xUe6EvksUGrvU1xGExkWwdRr48sWXs7Mf2N/CKHHc/Itw07sqQ2kfmtvDWjPM9aYtIldlXTV6dHP5SeFEH9AvOvPJDPWASlbwhYrBaeyL1ydk1fMxMsi31kYp6yylc8djYWIaoxNH0yrDCVy8+GqUmqr398+qeIEsUIwiZe1EWb1q00rRaaljHD5PHVEIeZMiRk8coiXtRc2dgyIlRjEaHVH3yAeYXjNV2QkFufUG1+44vSqeUZhfZZ//75SrhYtPiqv8lReCY/CZbaWGCRItvp4Ix8rfPSRn4noM1KJq/E014FqQw7VqvwL/XsLmH2hHLPgw+0/hJv6XQN3sbguBCA52i0JNfavX4LHgjLo1h+yIBjgd8wVtXwdfb+LThFVvGmMJPyzY8xEkkljuUpEnGpGHsl0Ammz8rBfhGE8hSd68V5lIGFjF9pghUoif5fvzw1zlxtg3R9Hnwdxm8hgqnUWMW4PaRcsg6r6lZMxiU7t/cDVewpfZjP/v7f/ddeFwtS0beXNvC3LB5XGV0PkyPvHi2Nr6fHd3hQxgno9TV9UFG6Knj9xT35b7WPA90HfpfvNmhQ43lC5h7oCrSTtnGUtlk/WbQTGv8inw5xBgeHRwUG3sg6AlRNR3hUtnmwIdhUFAZdq5GKQgTrmFbV9qsiGD9rCF4k1ohUftNxAfULaEIekCzgi4HqNfBl1b8K9L3efYBLfzR8TKLVajRAaZ4GbBqgtMgS1Nr2x39o/zRLUZQdrBuyRzieeyhJel0a39FRtzBjcadKv3hrZ6xKX6/LYmZbqU9V/LCCwSP1FUsYlnpsaaovyUr5qUyWvbmd8fRlDGtmm0eWb2UyDV1dJGmDgKXbGG0FHWdRZNcqtdwXxlHp1T1tHe+rx5c2Qv5R5tFG6Evq9pROqWD9WED+xtfl8XAzsLjpjdtqvHVMvTz6b0TCcvk5HcyS22cy5X0LsyEpR7ZiLj9ga1OknoioxNPGTc175i4mZj01KW0sGJCntVGu+C7H7q6ySbFYBOAFv+mkymNsH2iUVQw7aqPOCApcY+V/KitnhlzUF2G3taNwF1cZT/caVE110xZ3wMGAT7fYvO0VlRVxp6asxbhaXExdLBcpEfvi7asv7aXq7ifv80/fMU5VObAeh1FtfQtL7MAsJSjHVIQU142qCHVsOajGIo1bjbCpOL0aaXG2dfVIKWuseaDt1Qv7PInzWZm8II3mVd2o1LQa7TfV0XoiDxPJSvxG+RcQmFj96hbRD4sjgh32zKiipcvVqHphoIpnWICfp84Gk3yDWUAHbUdBPbRFmqQogUYSHwt/FMK/qBC388MMgbcYGi2KOqCslDkWayzh4GpZ8C5XLLI6stgIinaVQ1dAP2iGbhTOasqqeFI+LuIp7T4sPyzS7zEGhTQAPPnzH5Qv0DPzBVZMX+YhFueJ1q10QdX0zjrewRM270nnUYEJWxX7AoeevuUrj1KVdU6tFRVJ1xY3MFfFtD02mak49qLOBAxSRkvKdR9ZWFKKLWDwNxU9227fDrByGWqQf7095ERkmOlWp8s//fY//Rv2Wsgv24FVg2Ix1Ewc+BUqLfLfrPYaCdRgJ1gNL0CrJfNT0fFTMAt4gl9jtMHCkZIaz1B8VpRWZSkskDR3t1rjRjNqtdytmNj6KzqFgJvwCno2mtvg3iWRO/W3QMqaAKn+8gwFFDk4EezjH6qxzI9/3Mr631Lczun79XLMNAZ72Xguc3Xur/GE5+7h3W4DoOc6jLgeTNNOpzpnih/eOf0arFh50HLTduxU726SDq7XOC0E0jRe5RbAk3XxN32yt9mgfhcPlXeIrnZl+b08cf0gqT82ThuP8NtW4bUrt+Dk2cdaiXKYfwfrgyHz6VgLtquKYl0VQLfGnZ/M6Ep24HgTHo2FbXbaYfrfs+bvEYAIPm+vAKTKl0V9FmFNNZU6WoI0Ss5XK7GjNCGNgqKlQwZuR1mjHWn/dRSpdiShdUrJ0lH+AJX1WCUoFO47Snp1CjvDOE0vIleGeOlo/p/ngz1mGO0n/W5PPaLyU3++p5ozoj7goz/VY67seZWggTKCyBbz+jdq1Kk4McejAJFdfiqpYyxEp1gHYKuSICR56FVHpG2IAoFuAQIAYe5dG7sP5l06ZI+RJSzqD7YkRj52d+lbYbKfFfrWSXMPy1nYZm15NdwzRbjmcM87zYNZr99uPxLiKjWOksNWdF6HbwgKuXwgDeWHyRSH60/KlH/bjXKMdKLwsazPFARRlmDlO0BFvQ7Z6kqUxTig3wA3yC8n6J/q0oGUB4CJANUulopoqdfl4d8oF3PuqoKxjfGFrT7qZIC3iBvo226qp2WgU6pIuSKlxBjjmQJ2ThK0+B6IzMVvyiGp9JSheqnMtuRMzKVMqmP3Rb25GpK1jEbFstWFBOqqGZfocwKV0QfYYbFudnexDn+dogo7VdFVhV6AdQp5reRMOYtOKYO24HwGXLgBQsewaM2j/MoMUAM07QhN+HDP0GPSf8HKx+TBkG1fJYNhhYmw0pTXYe8H9NYgvuibpZ4VKK9tIqP0DFdM+6ycSqewSCufPbky5qTL2JpdFHqxU1XGDeq2gL9S6TYIG6OulLssVmyj5mKnrK3V1hg/G6NUm8il2jH2Oh1Zg8wuy/bpV0rpV74wNLqmWqMAeainR1Hsjg45n9MjwifFeVcIcKOMlQG7KNSlB0Ph1XZFUoIKdqnO3K1lombIRpQ1lVc1+p3Gsrid7r0BwUsERG+Bm5QDgYoRBwfL3aRCMZRARrJO2FZjRlA7P/4zAvRNv2THffPs9TfPXle/nzoC88tF3Nn4pyzA8yTA77ij90seH+Y3egJreyMwjPll9HmPYZ7SRjmb5ZQ/FwbDZOl7n2nsXYjym2kXsjRmEGH0BEMMl2Jo6hdUK3qs9Om0i4Yqlz5+KwTL9VW/FyKF+/WyXvGB1uJU1KL0q8szFZa7LkiY1UdlZXkUCcx8LQeHZd6aB5Y56lufmR4DomKGqEjIKUBMdRjh6zN21G3Xzm5JHDryHxnfTreYduoMAwyaWF2H/odeRoNlpL/m2YScr95QSeBl2JWyqBoCjF+RVsjzyJu4cplpeoN79+/qidFEB0v0V3AQPCvP5ZZ8JYHaVEggoY9gDYypqdvUl8Sn7F2VGcPCGGJV4jxmYYDWcOVnrFw8ule8Fl+69MkDdUMfr+g1bdZTTofwXSpCMDirlzE0s3CUwNBUdaZZgr7Ph6y/LAqIZ4GSc38UwmQKcSiu0ESx9lWBiX2jibWm2Ci8Zn4fsFp2DkQU2DrIoHvv6e9XmJp0tebQgTXJslna398fA8nmoMfjqTESc1D74E0Jnop0H09Dp9k+RoFQwu6vDrhbxvt7gKj8ao9P/btHKEHpaGetMizM1ZuAzMdZdlg3vnd8XJGucLtKqiCHbOR3kMw+fVY1T/H8HMcCU5p+e9Uq2rp5tYIp3ln6pmWehCkfCftB+/8mA5g0z3ieTQY2zqpTjLMhMIzN3CmQAX7Sr+nc6va88qPYQypGClfKENLB8XRo1ZkGtUeFYwDR8JM+MLwkFyWfVDnIrEPuVasvnlJV2zMioFLeW3t7UbwHWIplZVd5C7rdJgA7IiYBHukd3KN16/UXuFQ32+RoyOqIA3OEL0+foa3pA9LN+2/ePgWkGlUWjS/OYhHK8rNIpZ6hj5tXP5iUsUfsbrdWSVwWLsKeVLlGB8kgrNYpJd8AHUQBRieGqu2kEA9nfO9Dd+/B3vln3zqwPOVqfIsiA7f0sQvjcyiG4p8uKyK54FNnTDuEDVFTsEl4+V3FVR8q1Nb1GX1kGNWzi+rZmOf5mu8YVolS0cq5wtM2NQXlZxitJwUyapUDUfgWVgZOeeljaQrC16n+er2I8Bww2GZDsLNj49PsdLI4kedCY8cxNyNJytRtpWaBtCSUSNZIQdAsalZLjkbSv70uLcmBBkYjQsnbsERLckOpBWgtJX858J3/AzjZabE=", "memoria.py": "eNqtWM1uG8kRvvMpCr0QPOTSpLzCXhhwA8de7waxsUDsHAKGIJozTXKkmenR9AwtWuAjJECuuQQ5BjnmECA5BLBeKI+Qr6rnj5Kw3gQWIKmnp7u66quqr6pHKfXGpLaINUUmobywBxOWdkbbSheRpsTSdWUIryqHmdiSzovKrDVZCm1RxFtDB15VOT3IdaEps1SY3JRxQaYobGHcZDAY0QsNaXioklJHlvLKRAbPYbzGwv/8+Y+/n+LPH8hkMyrj3EKbEopg6Zg2tkj1mEpzU1Y44DAgMg5aJaa0U53FqQ5jm1EQ4k9oU5OVoqjNeVonwwmOf62d6GtCnjQO8v05QW4Kh2VeUDIN9dp80FO7voT0ITlTI8GiM0q0w+mpieJIi4xEE3YlekK/SEwWmYKXUGKAyphcTElitpqqjFILfK0IqbdDEAADAhiPGT9GWXS6h7hJ4i3+BXusBbopm2KKUGe2Nc04QBW7Mg7xEPD+1Ja2mOaF0ZeVKw05nWDSXNpCjEorbClglcn4qFxnJpkM3jbW8vR31m6x52UR7w2bwmdAbMZO+RlPZPCNYUwo1AU8ziHE57BBk4FSahCnuS1KugTAzdi6ZlTuoFwUZ9t2Ik7NYLApbEoTqueA1ybeDgar1z+8+BXNu02T1za8CoaDV98+f/cWLwKls3Bn1ZhUooutDNbaGf6fx6bItJOhLv1gXegPVkYhIimxajgYDCKzARTlLhjO4B2iSGyfk5pCjxKBNZWZ6ZuDoKJkUbyBURPeNoldFBeBrKkl8E9hELdZu+jSxplfg8N1fhGtkFKInfiDvjQTxgq68D5sSPUVgqVwgcdh8sNv3q1e/vLXwP0G3l7Zq/k7ZKNf/tgx97epR47yZidWR43ZZXHotH8flzvkkskCj8yQEGKb7n3vaBY4EUEbr5K5CU1e0rfyD3n3AJNbtUecOjWjxRLacfT7h2OtltN7E6S1Xjlc4ZXwaqYyQV+SmmDsvdGpiymIfK/uKyxaRlWaBym4ZUwx0jYr5884mF1VmJV2YRzPX+nEmdYRoLREh8YLzRvQNkbDDAP3IOtrJWvLFhsAUcqLydaUwdWYzoEduIyucCRJ3C5rOQDBBBGrskFOwUXbCmk2JotdHaPNlarPQG595/MUzMII9hgUOYjJlmknnIe8RwMrli/aKM8KHP+3x2EHnKRZB1WKPT4wuqlF7bLlROeAOQpuT0JBlWommTwBv2x4EKiz3z49S5+eRXT2/ezsjRrCLZ4OsbQJV87VTGN1NByfCmzgwOIOGWWv8Ly2NgnsFQvsYMJ897CYXZyfL+9JFG7EMu2x8I+QsauyQ6UzURBHOVOucu0kIuulwCx36r6GoD/dW7RBoDgRwvGM+aDDneMh3oCmuQwhNBEOcIB/J6tPRX+en+541Gs17Kl/7BxbZ1odkaXR4W5VHnIT+KBk5Va+gqIutmH4ou4BmtLlo897d8aFDDEKb6CIXZo0RwGUHiGxDgVkW8XwEvqDJkQZnJMovYdWQ7eZLWXtAzqRnP0fo9lzThfNagOPnSZ258kTFH7UVShHnF8zahnAyxBhn06RRxzTs5NJvyHIUpdulbbcfc/IPR5Tf7xP2zHI9QTIPeksklHaqenUw/oVqOd3/8y4v9vpg7COm9BL4/Lq7q/SC4Xc4tV9DrDLLYiVRqOP/36LnuDu77SGu38+GpF6gJs6UK/5pLi4+wvVlQq/4DCvcRJz5zYHt6rR6Bb9VrAfHr0m9PEfJDPeiCdixBMxFkse7fzqA9CsjEZqKQcwO18zOzfJSEqaTx7U7ScP2+6zj9K1aHYjMm5Yxp4Rvll0/LWk+Zyul+0OvL52p4XUXkGIq9IA+8Bwy07atRuerBQsmqDdqKd0ez0JdR6XwPqDCYbHGd3aq+NUQMHmo8BPwe2z83Ma8UHTKbXvzoY1xF9Q2zz2WkZu8wSIFiW/QpBqWJMxKhj27f8BS42zxOJNn5aH/FaGJ8DVmtM3c/rqs0H4inWQFv1WTmQMH4erAbOdkSBsQMSNwD5qNmdZo1bP0l7tGi4XT7+eLZsEFUmdfScaq99lo9HdnxIQCE5D8qwTw3cDdiDuBuhzSzNDaA9Pt9OXkkAImJvFk8YJT5ZsKyY6VTClOvVFkWWfg3C88k2miG0KB/LMpinrh35GUJSb26pAbEYo3LjX4b5kugry3HcivjAUJpY+WyP6WDOpFILtDDv3yDr040Go03VsHXfBeVJXh2FbRX46A6ZQ/NHobAJQehSJwIdtCpduNGTL1uOQdHuNJiA7BH0/dxDyaY9SwmlY9kjoE+Rz9FWzQeP97sA6HNnAfgAtajlLVlmIHpp0QVXvX6isMnu9ciaNcaFl1XydaS8Cu0MbehZ3R6pX1iVde4bohdsXNUsIowiNCKMgccqq9ioF0AXxSxf1c1NGKlxuuwKkSyOWDfoYYddP4J/PyEHLE6GnJHQxe1DUWOsFdv0IHVHLKe1uHZaVTvjO2RjDJ6FRCESXIXoGXZSO+5tAvSukpUDxxJW+BaKVBS9xVLYvWFIjv3+A39489mlWJEBX8QDjIkbx7JK+kQfByQvlGw4uY+eTZ1+fotFGmIeTEWERJ2t60bXx61C/sOhIwcnXC9AUKCLo6TEicHRLysOuTfSx36SOZ9yOih5JgO6l6Nh1k/fDv/kUBarLE/7wETiA9PFv6Ft498d/PdCiy1qvR02Ds16ycN2JTGTd1F4iEzb8WUk+9pjMf2PinpnPczNhe3AkmiYQUhzah7bUB3zCkFatWjQFX13QrjKuK2Y13bcsw3nmeR+C5AJzwgZq8F/zOi03", "estudio.html": "eNq1XFtv20iWfvevYJTuFplQ1M1ybMl0JnGcy8CZZOL09ACGkZTIksSYFzVJybIVAfML9mWB3X1bzMMusA8DLLAP87YPk38yv2B/wn6nqniTZHe6e7eBbpFVp06dOvdziu7De27kpNdTrk3SwD/aOaQfzWfh2K7xpHZ0OOHMPToMeMo0Z8LihKd2bZaOGvs1AIvhkAXcrs09fjWN4rSmOVGY8hBgV56bTmyXzz2HN8SL6YVe6jG/kTjM53abcKRe6vOjkySduV6kdZ8dNuXIzmGSXtNvP46idNloDMf9++1Ou9veHzQaUxZyH+/DttvZxbvvhbx/v8M6bncPr+ki7d/n+/yAc7wFM7wdsAOXMbwxx+nf7w17w1ELb9ElsIwO3L3haufBchgtGol344Xj/jCKXR43MLIinpjDyL1eBiwee2G/NZhwbzxJ++1W69vBkDmX4ziahW5/zmKdKDUGTuRHsXoHNcZgBLb027vThZZcJykPGjPPbLDp1OcNOWCe8XHEte9fmQkLk0bCY280iOY8HvnRVX/iuS4PVzv302i6nEYJ+BiFfTZMIn+W8gFGQcx0MfD5KJVPsaIQj66XTH123R/5fDEYs2l/H4PM98Zhw8POSd+BwHg8uGl4ocsX/d6AIBtXMUDpP6sdK+HjZQXNxqmFSIyB5Fu/TQeNfM/V5CwJKJtsxMz1Zomkbcpcl/jdnUrSOtOF3E4bztI0Ck1rmIbL0m5pDPZMWQySK1yGlPPdW2s7PSpthGetjV2kRLxwAkYD0yxOgGoaeYITVxMwpoFtHN4Po4IHiigrqpB0v+N0Rl22IfTKmj7Yx4Y+d5cR0Hrpdd/q9rJ9XT5iMz+lBdMl8bffxrPrFqKOuc9Sb87F8NE6U75eBFtodF3NCng4ywUcRiEfbCqZ1KiWVLb2Ns3/lTqwh+fAC6Wz6LcPaC5gi4aytr3WfFJYBJulkSDeiqY8XDvB0I+cS8zSoMZM+SvlUIUZqL3IkFO+SBvCLPpkRps2LABc7kQxE5yRfFLE75NeEcXVA9KZSnwSS3ItvUMFc+L7Ezpy5QhyaIsGZouSgPn+csM6wDeW9oUY4Un4YnqLHkWXuZjKOGiYKG5cZfJoAc0wAknBFp8kJ/rC1oRb6hGPyXpHURxIO4ZO8z/qDcwYJe9TcTTCFblezB2BHgTNgnCb98ocG2mMlOrB3vwKFLLQC5LlhhMkyVT93ODTLEm90XVDxTCFOUORKdCvtruSq8rUpLNmBz/potaIsrbRhVC3VZBiXA7cH41GQOWFo2iLCGU0QdASNr9HNG4qFWkEgiaXks6kuDugc2em27Z62CVJ2ZgPmTvmS8LXI2kpALLmkuQetYTkfD7mobu8zRWJDZWaPVpzFru9dWdRNkOVOTjOL/ZVubmvnb/iQ3Nm5Gex4N8015tXfG1x0g0Zy/HKiooCb7EDlaSAGK21yhB4y8zRFTzf4q26pRN4oZCgcqX34Tu2ScILkRPClVWo20LVdtPaVKeMZd3VjsPCOUvWffpvAu56TNMLZdnbxUGM5Y4mkqOMM6Rh+5nr2c8Ton3KL7SNBKNsegdVqXbFCmkihFTqktS67RuUI4k6Zwlfu8CnDeM1RdAyf6o26LYkdBpFfrKsurZcvRsLFQ21qqPKDtXboqvqVGsmWT5bu9eqHKqq6LRYKW0FvnrQskV29mCRxQl2xQmyVOigs9pZ7Rw2VdJ/mDixN001Kk3smhdQaRGwae1oqV6SWn9ZSycx57V+bZKm06TfbDpuaH1KXO5789gKedoMp0FTAP2mZbX3WlarOZx5vivHrCByZz7HipopUTXBryhMmj8PJV+wAGl80vyUBM3aaoVDCOJxiqYsn6h2wBuMWPNcuwZGo/LRNDHg+CxJ7Br0sSYmqZBKxDQApBw1l6WsMcdCvqhlC6KwdvQeajaL2WFTwh2twyNLBqOO3nr8hiU51FbUMqQTME+i5FaM2Py6dvQkdjzfv33fiQcYP70d4McZc7HXMX7iqELZYRNM2WAO+LV1wnUl03wivpZvo2Zh17Wjv//Ln7TT2Y3293/674KcEgrKl7ax26cCWBSltaw63X4cAlzAxL0oBqR60nS4R+MOlmMVfuB6HY5lT7LHW7dwYxYg6XRY7egZHr/8mZ5vhYZDCQH5ln40PUYJnrBYS6W6FGRJnt7JWYcFwXbG/s+//sO/acdf/oxww342dx271jJbtaPnVMbx7ecATHtfQJ2gEPNddivYgYA6ZW6keTc/WnewHcCNEjQin3Ur1m7PbO/XjrrN3VtBWuZ+i8wh9oa3UzdCEQldZ2VD/WnGIzvfzveTBfk/iPNOnmc4pAAq+2U/mTuiwC7oG7MQPIFX5H709z/9+zpcESpqmnDTIPZrCsZKWtirZGGqDVOkopsp8JY0uRrFBqJpM4E4r/otraW1KTNTmPf3c8dRHFcErc1xisebozK2Fe6axkSMvcUj5X5cBOw11afxOEorPvwJIm4Dg2zT9oX9pXAPxzCSuASwgfQK5RHMPfK//GUchVv8t5DfJffJln6ccZ+n0XaYCciDaf/jX7VjNq0El8I5Zz+VGC2DKQ4s47P2QGOJ9v7lu5MTbRRHgVYXQbM+yOaXb+Khlx5DlDEYtSoDZWHYUZPNCijCcQnLi9P3z0+hwEiWt6HwxVTSLMCqy5+9e3L85ivWl+CqCN5FUXASzr04CmFq6VYkvJhPmmsLJLYdHDVJtW/sxD5yI2cmZiCn+PoMsnLSKNYTY6Cgnr86PTmzl/Dm/fO6tNYPV3xojf1h3cwGSEdhEtVBGQC8m2ziwhQZAtBMRYqgBlUuQMOUDKhRCv0YYjL2q8GJRzQwDHxgCPhqVIT3gjjxKqdW2SF+9+r4xF6uk9+vvzh9qiGa8zjSMKjpCHZOpD3Ufvv25IWx/XhykRrRdCwztReRG6Wm9l7keZ+S0kr4JGs0XGDV86d/pGa1xjOL0PTvQySjpvZ9GHPmm9prD6ldhMU7mlpuDX04Dyx+Sr8ccV6OGLczWZKXeKWNCmCkGBTVIysafgLkm6e/BVQqontU2vYD1C3mCZxr17WS1Afo2ftTTY1++a9Q6z5DRRSUyPAjt/2h11IknL55lrG11/q2AtX50O6VoILZdQbZ7n0raCgpR78uk0kN2q9NZlAP4Cq0pF9/zaYMYVUTY5grqwuIhqPXqPvDiISq2tBaDGg0oOFYCCGOOJpOvIu5i9e4dMBCq7KVHIGSMkpwPsEZgqknkh+k29h2/OUvDHqkciBtEsUhcnNGCOeey6MPyHS8kMVWMN0Fxj/QmFQqQUUONobldpH3l8Hw/rf/FBAQ54cp5VvMmoZjALwKEDRD4oiY1ORkfZXbfCz0iMd2yK+kv7R+4MMXp+/UuL5kId2ZeCzpp/GMmyRyHs85LOMKkfLpbDTisZhawUVk6FDgpm+9BfffUatSf83SCbUedHkdU8yYHcMYlBedIa7qXhjy+Ad1ZYPHl8IjlNFHs3Q6g09G4D6jLrktST979+JpMVggRgDhUI0pCFaQT45Pzp57fuA574u5EiUioGPY4qHomtt0wsFO7h2pprKwCguOJ6jp9HypCwfrcwLKXWaCKpyXOHxG77oxEONWkZGUQMQp9NZCXjnlmKZBzIMS2NvX705evwCWmJGXzoig9hqBw/WjSLPFKotCg9yZ1q8FAt0wUUzuGpbUUJ5tCLUsb4dgNKVO6JwfswCb6shSC2k1S9IidG2z02rltDupL1BVwqmODcytvAO4Yv4zFLgkOSECGqY2wTvkLCnfNnY25ZBX2+pkG49jr8zaF3h9yf0plLvdMneRji86vNuWD53ObsfJJIMAqtPi/AgjP4rKpvKaJxO9xB4UPPwFjwKextd6p4XjG2ZJ7EqpqEZjvl6+iFkZoj0exRblYpTGWgu7Iczm7atmJ5tDjQbWSzzy6AWhAiSndMIDr0ToS7wm0wmYfErCgWKNxD84chf/7O6a1l751LS8UN9ZWEL1LOuFM38NVwcYZqHlsCStkIgxaU9WwKZk4GToYM/uvkn/Ke8L2Hzb2Avu3nZ/nzFs2y4jwCIg2EGYgzOGqGa+bwZwN+qRCRSJvVwhoYicy7LF0TvMAPl8afRptOhi8CX0ad+kFFZqjsRGOTuhYnPmSd0+w8HyA8jsTYwXCaCe2aZLqYWYLGV3mBXjxKFn3EGoid9CCfT6L20I+d4waQqUzbGfjpp1YyDJEjuUNhYwxLrRLBRM0kRXQ7+kzqYmST61l6op0T8vpG6ed8xds3thFhKBNpltq2WS/C/MrD+hFnVc1+xaWNYzoQC0bDhy97HM6pnWAS2Vy6jMyJsUci1r7cGoAHHe2MXGYvFe61GL9gRGGJL4V67PexYlYrvY47xr9sxGm9aORr3eo0cmydYC0e2eOWJ+grWig1FaZ+2a5y0s68lV0nSwbk/sKxetzi8vBiBa2AC5b2LxS77QT89bF9I0qK0eoky9tk/P2xdiKCtbhU1YlnV63rkwrGDmp97Uvz5zmM9i/SWCpBesY+1eyOEy1l1BAo1WEDc6D16a9G+j+wDIyLwry3oXyoZKxQKGH108liGkT/q+CfKqhGJP7OyN9Hu0zlhu2eOhtUdt1lzDhrA13aBbB2s2deEX4fBib/FDFCOgkhDpCmQBr3HJT4Jpep3bznDDQmlvwoNCdY4gxfXIPlqCmMjyEvLSRvZydklBypVjsLNLkRKHigAgisDmAMkFf0ohmVIcInIwxPD0GiiGxbBBGYB/LanexVxQ0G+sRJnMoRnYZqyCwk/gzsG+ehPizywEM/WhsVpRLoXoDc5GixKfR7AErrMbG7bBfbvdETYtua846tiEaczTY3FhUApqfxAFIFhsmEkGJLK0bSCDl7bM9dhCT6xrEzmAFI10eeXJhUkAiXVjPGhbB00xk7KQEgJrFM0fZMEP2a2BCJgjYTa7ySepQ8dt7pcHBChbMwDHWjwUMMjmdWZIeCdKdG48cE3Hui5m1cjNwxxmHV4eCUkHvBO4IaVHd5rYFWl9bLtNJD7idURvD3ryRerY2zj6JEOZlKkuc51cAcsWQvVZAhOheJbaLXNutwZ3qfl3392LrBkS9GfwnBb1ZWBeMgmyC/UazB/aY4ul2H0IZUwKVjnQu3SQ0rRomD1Wv3Ki2e3fsazZJQXM5fQVygLYb/S6uIqqG5ZMHt+/PrU/frMU/BZJsZ4iKY1OI/pm6wxbh2PdWGl4+PLncDzzUWv97a/aN8v5FqD5l/+IEQB4cjiMj75ZQucA9BzZgKt3MP3lnzUavN42eFMZDD4+JGvWiVJL1J3J449y39LQSpakycd+vQ7zZ8l16Gi5KPkCFViij4wlJKCEEtvsinnIK3nqTPSRuYR4JpHbr788efKsXpgzCp7LlcMICAJVgyLqrFY76xslk+hKp3ucUuR2WOgmtmjdnNPUxYA0auT5XGQyAxTFukpxtWgkwQ3olchsrAkjuo2lgB8NhjFnlyvl7GnMkBRl8qQEA/IUTVpL9WjtOt3M1gv1GKujq2yEfp7QOXRCCJsWas0hhZT5Rgkr1SfH6hu/j5tdY62iOu1W6wEXuLnbzJCtvv0o9RQbwJaMpQxrKJGiORcjK8wUud4aQGlisJYQrmQQgvGIFYO7A1K0kSlH2zL8CIXbDFlXcIwdUNgIsQ9EcAE6EqZt23Vqj9U/f85eJ17dEJFClBprFcsZ3KyLvKqoRGQHHInYwdBx9kwwbzwJeZL0rV7PpM8sffnW6knOVbZW/bo6neirdwRPIElR5arGQmnTRyvEMrlTkdmTYAbrYSsIhZcJvNC6FuzOXdJ1ww5CVTcVg3CfVNBVR4C0KBEGRYkwILUT3faKd6rXlfLAF1KDRoBbPg/HKaSqMOXHf5KBvKaJ/BSZGYjrBfu8/o6DJOpmvUTFy2I8vPZi8fskgap79PQ7PhYjZ1FMXRhqYj33qXWPhxcK+h0fqpFj2VCipyhGcU0LmT9zmXqSUE9h33I7FoqO2dso4R/e1y8kkZUjJlGc6jozh4Z9pAvCZXB4Az9h0Xe4xsODA+Pbg4PG2uywMqvOX7gch1xOeSdjqYRw7oiVF7bgquX43vSJmKFoK7BkfBzaeWvGgXtKueoj6HV5f1CnLKvsOyRmGBy8k8P1+gecXRNQUYh9UBrqOCa5Ll2CGhV1KLd+hirXE8AZ5ZlELx5nT/3yEVESKKzyaCr6U9NEYCP3DLd7BgejLP42Z1JOaLMVqh1HBJOLAr3Z15/2PQVTnhT3QacIT5mPqwuGkfnx9AckEToBUzZRAaYLEuaFiQTPw7m84doIAOvuonBW4hag/rguvrCp9+t0nVcf4HhFZ+lwb7dlVJAXZJBzqEPX65sUFEa7vvvj+uEw/9jgiPKDY+YyTThCzdZmoYzlFs08i6DHYIUW8BtsG5qazzTRPqZBN4LBWvW+UIH1I9EmsmON+ITYwONppG5Oj+oPBdSpIFY3+tKvyFwdVbqoviH0U9n7VJng4PYAK9kGFNuvcJ74PpaK7zu0zCaQaMcnDFnF0D4alliaRuOxL9XAHFp0kwxFsOY4mcgrqjlq5RQqsUk4lz2j12xKzvoXpayB/SSO2TXgxK9ehBfjcfEMS+oXb8o1B999Fwj7AnaiRaQwcsQwxAAVBXLArN+vP8yK67EorrMU0lBRSGVc56jPafEFNbJ0/Tw0nQs4iY+H4nrby66lS1fM3yydFV3Vekg/w5W8ufxoWJ8i1BkiSVSNKooSMvyU+EruJMyYgbJHeZbwgmzjHsuyLpnGEIbvvlMP92ybGeoZJYjL38xS3eoaA2ZR+z7VDTH6KqRBS2wkhCTIYD+lQ+WPvL5Wh8qu17bDDb8LVpeqHuV1olCU7fcoaKrDSmZcJfb5xS1aVVWlK1Qps2SiRwYkiZeM2Cv76MqSn/FnTu/KuLWLcC9T1s+f1zcoUl+heBstBv1rdLh/XjxfFBwN7KNA7CKcgg12qIJglW+X68fVelu6KPfM6sxTlnjOZu63P9wfjkZmvp3Mykp/7iAHSk1r0+XTdPJD7KW8L4sRI4/KV1U2yVT2ypKd/jci6ekgtSXffUVWtsp7tsLf2XXVaKyX7lw2VTH7W4R1LcxCOJURHH4ymqLonrKxiL1FB9YeKhVQyYIaj2x3e5RDyJdB5m6SStQs7KPFtvAqMBnCjiPDXQ9mcppcT74Rxk/meCAguu7R6+KEdZOylP8jcow7eX0uv+66uI3XRMgyl18RNfz1ULb6mn2cn9inegskMyQKjXGU3pnQeJSNZqQ5FKrp+yjULj+7G6ZAU3tz0rEWpmxcLlCHvHxgtR5RQylL/Nd7UynAU+sa/948BHCvJxtC5eZSutYjKtyApOKcmciSSyezkBR4yH5NcIOC1e9mwZAqVtUKNPlPieHOTKEsC9FuKCUJpEaVJKwMfEsKl8UKlcUVgrxL6Peq71vFXw5CVXBjJfdx0nhtnywD2yklvVU6pJcI7XtfkRYPtsKU6SK3Xop4ii6Vmt+x8UbyvmXjDZj1jXfyMFLuZRRXlqo9ru5LRfVaAOYxyxKR4D18tjLFbTD0Ob/8o8xO+cav1E6R5nFn0+RXhtNlYJUimoxKgaVimi0uYkUwUo37r238DP6/ic4obA+qB1Bhl2jO9GayaTj5/bp80MWxTPihzImxW6tnVqekcYKcwN5ySw91ohD//btTve7RpyZN+uyEVrjRVUhlil135Id84oMUTAiydNLyzZBGXxPdcBnTluQmmfjawN7+hcHdLfWv/ahk/auVvF9zGkXI8Quzc1NbXBlTWHjG/ZTpqnZHqVEu4d1NX30b92nvHV20b8VGUDvqDYtrZdkd1cv94XrsjT+IfvOnRFTc4he8zDvDRUBAdZpfSL8ZEnOsOfNR0soeNoudiTePks+flyppW+sAb/l6buOTKfPXff5nlr/sKn/GVf1s68LArvILh22EqsNd8utEp6/6jDVwyUAoSOAl8DgIbVTF4VcWcap3Li1WwuaNekN2vyUuxaZfGjGL/otswpcC5oWVIE/XQUOl2166Sglssmz1OXX967iw0bz/GZY+KhswvRR9FEJ+Prp4+FE7FH/libp2lOUaFg49hdEY9DcoYvIj3FW5U8aEa12jXZBJ9N5TyH8dreVKc7R1f4n998I+4LrOOFnDW4awn+iwcKYSMxoVvKbCZG7/ngxfr8/rxufP9ZTuMohkdaVymwzndlkyAhiK4YWufmkfybWXt63FNqKrr2K02l+mEpS4tuvG1txClhAq31XfgaUZ+TxlU+IeXa6kGaOHQr+KD+w3mqU8xcBau0l261a5iclrJ2oNGZQ85X+GdNgUf3902JT/i4f/BaU93Xk="}""")
for _n, _d in _FILES.items():
    open(f"{APP}/ap3d/{_n}", "wb").write(zlib.decompress(base64.b64decode(_d)))
if APP not in sys.path:
    sys.path.insert(0, APP)
# Si ya habia una version cargada, se cambia por la nueva SIN perder los
# modelos que ya estan en la GPU (antes Colab seguia usando el codigo viejo).
_old = sys.modules.get("ap3d.engine")
_eng = getattr(_old, "_ENGINE", None)
_app = sys.modules.get("ap3d.app")
if _app is not None:
    for _k in ("inline", "web"):
        try:
            _app._SERVERS.pop(_k).close()
        except Exception:
            pass
    try:
        _app._SERVERS.pop("tunnel").terminate()
    except Exception:
        pass
for _m in [m for m in sys.modules if m == "ap3d" or m.startswith("ap3d.")]:
    del sys.modules[_m]
if _eng is not None:
    from ap3d import engine as _e
    _eng.__class__ = _e.Engine
    _eng._ensure()
    _e._ENGINE = _eng
    print("Código actualizado; los modelos siguen cargados en la GPU.")
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip() or "SIN GPU: cambia el entorno a GPU")
import psutil
print(f"RAM del sistema: {psutil.virtual_memory().total/2**30:.0f} GB · CPU: {os.cpu_count()}")
print("Proyecto listo en", APP)

## 2. Instalar dependencias, TripoSG y Blender 4.5 LTS

In [ ]:
# @title 2. Instalar dependencias, Hunyuan3D-2, TripoSG y Blender 4.5 LTS
USAR_CACHE_EN_DRIVE = False  # @param {type:"boolean"}
INSTALAR_TRIPOSG = True  # @param {type:"boolean"}
# @markdown TripoSG es el modelo abierto del equipo de **Tripo AI**: formas con más detalle y bordes limpios. Se suma como segundo motor; si no instala, sigue Hunyuan3D.
# @markdown Si lo activas, las piezas compiladas se guardan en Drive y la próxima sesión se instalan en segundos.
import os, subprocess, sys, glob, shutil, time
t0 = time.time()

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        if check:
            raise RuntimeError("Fallo: " + cmd)
    return r.stdout

# --- sistema (Blender sin ventana necesita estas librerias)
sh("apt-get -qq update && apt-get -qq install -y libxi6 libxkbcommon0 libsm6 libxrender1 "
   "libxxf86vm1 libxfixes3 libgl1 libegl1 libglu1-mesa libopengl0 > /dev/null", check=False)

# --- Python
sh(f"{sys.executable} -m pip install -q 'gradio>=5.9,<6' 'diffusers==0.32.2' 'transformers==4.49.0' 'accelerate>=1.2,<1.6' "
   "einops omegaconf trimesh pymeshlab pygltflib xatlas rembg onnxruntime-gpu "
   "opencv-python-headless ninja pybind11 psutil hf_transfer 'huggingface_hub>=0.25' sentencepiece protobuf")
print(f"Paquetes listos ({time.time()-t0:.0f}s)")

# --- Hunyuan3D-2
if not os.path.exists("/content/Hunyuan3D-2"):
    sh("git clone -q --depth 1 https://github.com/Tencent/Hunyuan3D-2 /content/Hunyuan3D-2")
sh(f"cd /content/Hunyuan3D-2 && {sys.executable} -m pip install -q -e . --no-deps")

import torch
cap = "%d.%d" % torch.cuda.get_device_capability(0) if torch.cuda.is_available() else "8.0"
os.environ["TORCH_CUDA_ARCH_LIST"] = cap            # compila solo para tu GPU: mas rapido
os.environ["MAX_JOBS"] = str(os.cpu_count())
tag = f"py{sys.version_info.major}{sys.version_info.minor}_torch{torch.__version__.split('+')[0]}_sm{cap}"
cache = None
if USAR_CACHE_EN_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    cache = f"/content/drive/MyDrive/ap3d_cache/{tag}"
    os.makedirs(cache, exist_ok=True)

def have(mod):
    return subprocess.run([sys.executable, "-c", f"import {mod}"], capture_output=True).returncode == 0

if not have("custom_rasterizer"):
    whl = glob.glob(f"{cache}/custom_rasterizer*.whl") if cache else []
    if whl:
        sh(f"{sys.executable} -m pip install -q {whl[0]}")
        print("custom_rasterizer desde la cache de Drive")
    else:
        print("Compilando custom_rasterizer (3-5 min)...")
        d = "/content/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer"
        sh(f"cd {d} && {sys.executable} -m pip wheel -q . --no-build-isolation --no-deps -w /content/whl")
        w = glob.glob("/content/whl/custom_rasterizer*.whl")[0]
        sh(f"{sys.executable} -m pip install -q {w}")
        if cache:
            shutil.copy(w, cache)

d = "/content/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer"
if not glob.glob(f"{d}/mesh_processor*.so"):
    so = glob.glob(f"{cache}/mesh_processor*.so") if cache else []
    if so:
        shutil.copy(so[0], d)
    else:
        sh(f"cd {d} && c++ -O3 -Wall -shared -std=c++11 -fPIC "
           f"$({sys.executable} -m pybind11 --includes) mesh_processor.cpp "
           f"-o mesh_processor$({sys.executable}-config --extension-suffix 2>/dev/null || "
           f"python3-config --extension-suffix)", check=False)
        if not glob.glob(f"{d}/mesh_processor*.so"):
            sh(f"cd {d} && {sys.executable} setup.py build_ext --inplace", check=False)
        if cache:
            for f in glob.glob(f"{d}/mesh_processor*.so"):
                shutil.copy(f, cache)
print(f"Hunyuan3D-2 listo ({time.time()-t0:.0f}s)")

# --- Blender: 4.5 LTS (el mas nuevo estable); si no se puede, 4.2 LTS
if not os.path.exists("/content/blender/blender"):
    for ver, serie in (("4.5.3", "4.5"), ("4.5.2", "4.5"), ("4.5.1", "4.5"), ("4.2.3", "4.2")):
        url = f"https://download.blender.org/release/Blender{serie}/blender-{ver}-linux-x64.tar.xz"
        r = subprocess.run(f"wget -q {url} -O /content/blender.tar.xz && tar -xf /content/blender.tar.xz "
                           f"-C /content && mv /content/blender-{ver}-linux-x64 /content/blender",
                           shell=True)
        subprocess.run("rm -f /content/blender.tar.xz", shell=True)
        if r.returncode == 0 and os.path.exists("/content/blender/blender"):
            break

# --- TripoSG (modelo abierto del equipo de Tripo AI): forma de alta fidelidad
if INSTALAR_TRIPOSG and (not os.path.exists("/content/TripoSG") or
                         subprocess.run(f"{sys.executable} -c 'import diso'", shell=True).returncode):
    if not os.path.exists("/content/TripoSG"):
        sh("git clone -q --depth 1 https://github.com/VAST-AI-Research/TripoSG /content/TripoSG", check=False)
    sh(f"{sys.executable} -m pip install -q peft jaxtyping typeguard scikit-image", check=False)
    # diso se compila con CUDA y necesita ver torch: sin aislamiento de build
    r = subprocess.run(f"{sys.executable} -m pip install -q --no-build-isolation diso",
                       shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        r = subprocess.run(f"{sys.executable} -m pip install -q --no-build-isolation "
                           "git+https://github.com/SarahWeiii/diso", shell=True,
                           capture_output=True, text=True)
    print("TripoSG:", "listo" if r.returncode == 0 else
          "diso no compilo (TripoSG desactivado). Error:\n" + (r.stderr or r.stdout)[-800:])
print(sh("/content/blender/blender -b --version | head -1").strip())
print(f"Instalación lista en {time.time()-t0:.0f}s")

## 3. Comprobar GPU, Blender y modelos de IA (carga todo en RAM y GPU)

In [ ]:
# @title 3. Comprobar GPU, descargar modelos, precargarlos en la RAM y cargarlos en la GPU
CARGAR_MULTIVISTA = True  # @param {type:"boolean"}
CARGAR_TEXTURA = True  # @param {type:"boolean"}
MODO_CALIDAD_TAMBIEN = True  # @param {type:"boolean"}
CARGAR_TRIPOSG = True  # @param {type:"boolean"}
CARGAR_TEXTO_A_3D = True  # @param {type:"boolean"}
# @markdown Turbo (5 pasos + FlashVDM) siempre se carga: la forma sale en segundos. Con la casilla de arriba también queda cargado el modelo de 50 pasos para el preajuste "Maxima".
import os, sys, time
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
sys.path.insert(0, "/content/ap3d_app")
sys.path.insert(0, "/content/Hunyuan3D-2")
# Hunyuan3D-2 necesita transformers 4.49: con versiones nuevas su codificador
# de imagen (DINOv2) no carga ("Missing key(s) ... main_image_encoder").
import importlib.metadata as _md, subprocess
if _md.version("transformers") != "4.49.0" or not _md.version("diffusers").startswith("0.32"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.49.0",
                    "diffusers==0.32.2", "accelerate>=1.2,<1.6"], check=True)
    if "transformers" in sys.modules:
        raise SystemExit("Versiones corregidas. Ve a Entorno de ejecución › Reiniciar sesión "
                         "y vuelve a ejecutar la celda 1 y después esta celda 3.")
import torch
from huggingface_hub import snapshot_download
from ap3d import config
from ap3d.engine import get_engine

# Token de Hugging Face (opcional) desde los Secretos de Colab
try:
    from google.colab import userdata
    os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN") or "")
except Exception:
    pass
if not os.environ.get("HF_TOKEN"):
    os.environ.pop("HF_TOKEN", None)
import logging, warnings
warnings.filterwarnings("ignore")
for _n in ("hy3dgen", "hy3dgen.shapgen", "hy3dgen.shapegen"):
    logging.getLogger(_n).setLevel(logging.WARNING)

# ---- reloj de pasos: duracion de cada uno y fin estimado ----
_T0 = time.time()
_RES = []
_EST = {"Hunyuan3D-2": 120, "Hunyuan3D-2mv": 50, "TripoSG": 45, "Texto a imagen": 90,
        "Precargar pesos en RAM": 15, "Cargar modelos en GPU": 110, "TripoSG en GPU": 25}
_PEND = [k for k in _EST]
def paso(nombre, fn, activo=True):
    if nombre in _PEND:
        _PEND.remove(nombre)
    if not activo:
        _RES.append((nombre, 0, "omitido"))
        return None
    resto = sum(_EST[k] for k in _PEND)
    print(f"[{(time.time()-_T0)/60:4.1f} min] ▶ {nombre}… (despues faltan ~{resto/60:.0f} min)", flush=True)
    t = time.time()
    try:
        r = fn()
        _RES.append((nombre, time.time() - t, "ok"))
        print(f"           ✓ {nombre} en {time.time()-t:.0f} s", flush=True)
        return r
    except Exception as e:
        _RES.append((nombre, time.time() - t, f"fallo: {str(e)[:120]}"))
        print(f"           ⚠ {nombre}: {e}", flush=True)
        return None

print("CUDA:", torch.cuda.is_available())
P = config.detect()
print(config.describe(P))
print(f"Inicio. Primera vez: unos {sum(_EST.values())/60:.0f} min (despues de reiniciar, mucho menos: ya esta descargado).")

pats = ["hunyuan3d-dit-v2-0-turbo/*"] + (["hunyuan3d-dit-v2-0/*"] if MODO_CALIDAD_TAMBIEN else [])
if CARGAR_TEXTURA:
    pats += ["hunyuan3d-paint-v2-0-turbo/*", "hunyuan3d-paint-v2-0/*", "hunyuan3d-delight-v2-0/*"]
paso("Hunyuan3D-2", lambda: snapshot_download("tencent/Hunyuan3D-2", allow_patterns=pats, max_workers=16))
paso("Hunyuan3D-2mv", lambda: snapshot_download("tencent/Hunyuan3D-2mv", allow_patterns=["hunyuan3d-dit-v2-mv-turbo/*"]
     + (["hunyuan3d-dit-v2-mv/*"] if MODO_CALIDAD_TAMBIEN else []), max_workers=16), CARGAR_MULTIVISTA)
def _tripo():
    snapshot_download("VAST-AI/TripoSG", local_dir="/content/TripoSG/pretrained_weights/TripoSG", max_workers=16)
    snapshot_download("briaai/RMBG-1.4", local_dir="/content/TripoSG/pretrained_weights/RMBG-1.4", max_workers=16)
    return True
CARGAR_TRIPOSG = bool(paso("TripoSG", _tripo, CARGAR_TRIPOSG and os.path.exists("/content/TripoSG")))
def _t2i():
    # FLUX pide aceptar su licencia en Hugging Face (con tu token en Secretos de
    # Colab: HF_TOKEN). Sin acceso se usa HunyuanDiT, que no pide nada.
    try:
        snapshot_download("black-forest-labs/FLUX.1-schnell", max_workers=16,
                          ignore_patterns=["flux1-schnell.safetensors", "ae.safetensors", "*.md"])
        return "FLUX"
    except Exception as e:
        print("           FLUX sin acceso (necesita token):", str(e).splitlines()[0][:90])
        print("           -> se usa HunyuanDiT para texto a 3D")
        os.environ["AP3D_T2I"] = "hunyuan"
        snapshot_download("Tencent-Hunyuan/HunyuanDiT-v1.1-Diffusers-Distilled", max_workers=16)
        return "HunyuanDiT"
T2I = paso("Texto a imagen", _t2i, CARGAR_TEXTO_A_3D)

eng = get_engine()
paso("Precargar pesos en RAM", eng.preload_weights, P["preload_weights"])
paso("Cargar modelos en GPU", lambda: eng.load(want_mv=CARGAR_MULTIVISTA, want_paint=CARGAR_TEXTURA,
     want_turbo=True, want_quality=MODO_CALIDAD_TAMBIEN))
paso("TripoSG en GPU", eng.load_triposg, CARGAR_TRIPOSG)

print("\n======== RESUMEN (mandame esto para revisar) ========")
for n, sec, st in _RES:
    print(f"{n:28s} {sec:6.0f} s   {'✅' if st == 'ok' else ('—' if st == 'omitido' else '⚠️')} {st}")
print(f"{'TOTAL':28s} {time.time()-_T0:6.0f} s")
print(f"GPU {P['gpu']} · VRAM usada {torch.cuda.memory_allocated()/2**30:.1f} GB · "
      f"TripoSG {'si' if getattr(eng, 'tripo', None) is not None else 'no'} · "
      f"texto a 3D {T2I or 'no'}")
print("Todo listo. Sigue con la celda 3b (prueba de 1 min) o con la 4/5.")

## 3b. Prueba rápida sin página web (opcional)

In [ ]:
# @title 3b. Prueba corta SIN página web (~1-2 min): confirma que todo funciona y mide tiempos
import glob, sys, time
sys.path.insert(0, "/content/ap3d_app")
from ap3d.engine import get_engine
_img = sorted(glob.glob("/content/Hunyuan3D-2/assets/example_images/*.png"))[0]
print("Imagen de prueba:", _img.split("/")[-1])
_t = time.time()
_last = [None]
def _p(f, m):
    if m != _last[0]:
        _last[0] = m
        print(f"[{time.time()-_t:5.0f} s] {int(f*100):3d}%  {m}", flush=True)
_r = get_engine().run(_img, steps=5, octree=384, faces=60000, tex=2048, rig="auto",
                      tex_rapida=True, progress=_p)
_i = _r["info"]
print(f"\n======== RESULTADO (mandame esto para revisar) ========")
print(f"Listo en {time.time()-_t:.0f} s  ->  {_r['zip']}")
print(f"Tiempos: forma {_i.get('segundos_forma')} s · textura {_i.get('segundos_textura')} s · "
      f"Blender {_i.get('segundos_blender')} s")
print(f"Detectado: {(_i.get('clasificacion') or {}).get('tipo', _i.get('rig'))} · huesos {_i.get('huesos')} · "
      f"pesos por {_i.get('pesos')} · malla IA {_i.get('caras_alta')} -> {_i.get('caras')} caras · "
      f"cuadros {_i.get('quads_caras')}")
print("Archivos:", ", ".join(sorted((_i.get('archivos') or {}).values())))
print("Clasificacion (medidas):", _i.get("clasificacion"))
print("UV:", _i.get("uv"))
print("Avisos:", _i.get("avisos") or "ninguno")

## 4. Estudio

In [ ]:
# @title 4. Estudio (dentro del notebook)
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_inline
launch_inline(port=7860)

## 5. Abrir la página web con contraseña

In [ ]:
# @title 5. Abrir la página web con contraseña (para el celular)
CONTRASENA = ""  # @param {type:"string"}
# @markdown Déjala vacía para generar una al azar. El usuario es **ap**. La página deja de funcionar al cerrar la sesión de Colab.
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_web
url, user, pw = launch_web(CONTRASENA, user="ap", port=7861)
print("Página web:", url)
print("Usuario:", user, "· Contraseña:", pw)

## 6. Guardar en Drive

In [ ]:
# @title 6. Guardar todos los resultados en Google Drive
CARPETA = "AP_Model_3D"  # @param {type:"string"}
import os, shutil, glob
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
dst = f"/content/drive/MyDrive/{CARPETA}"
os.makedirs(dst, exist_ok=True)
n = 0
for z in glob.glob("/content/ap3d_salidas/*.zip"):
    if not os.path.exists(os.path.join(dst, os.path.basename(z))):
        shutil.copy(z, dst); n += 1
print(f"{n} modelos nuevos copiados a {dst}")

## 7. Memoria

In [ ]:
# @title 7. (Opcional) Liberar GPU / ver uso de memoria
LIBERAR_MODELOS = False  # @param {type:"boolean"}
import gc, torch, psutil, sys
sys.path.insert(0, "/content/ap3d_app")
from ap3d.engine import get_engine
e = get_engine()
if LIBERAR_MODELOS:
    e.shape = e.shape_mv = e.paint = None; e.ready = False
    gc.collect(); torch.cuda.empty_cache()
e.log_mem()